In [5]:
from google.colab import drive
from pathlib import Path
import torch

# Mount Google Drive
drive.mount("/content/drive")

# Main project folder in Drive
DRIVE_ROOT = Path("/content/drive/MyDrive/CSE499B")

# Required files
KEYPOINT_FULL = DRIVE_ROOT / "keypoints_full"

SPLIT_DIR = DRIVE_ROOT / "split_v4_no_leak"

TRAIN_FILE = SPLIT_DIR / "train_manifest.csv"
VAL_FILE = SPLIT_DIR / "val_manifest.csv"
TEST_FILE = SPLIT_DIR / "test_manifest.csv"


print("========== GPU CHECK ==========")

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


print("\n========== DRIVE CHECK ==========")

print("CSE499B folder:", DRIVE_ROOT.exists())

print(
    "keypoints_full:",
    KEYPOINT_FULL.exists()
)

print(
    "train_manifest.csv:",
    TRAIN_FILE.exists()
)

print(
    "val_manifest.csv:",
    VAL_FILE.exists()
)

print(
    "test_manifest.csv:",
    TEST_FILE.exists()
)

Mounted at /content/drive
========== GPU CHECK ==========
CUDA available: True
GPU: Tesla T4

========== DRIVE CHECK ==========
CSE499B folder: True
keypoints_full: True
train_manifest.csv: True
val_manifest.csv: True
test_manifest.csv: True


In [6]:
from pathlib import Path
import numpy as np

KEYPOINT_DIR = Path("/content/drive/MyDrive/CSE499B/keypoints_full")

npy_files = list(KEYPOINT_DIR.rglob("*.npy"))

print("Keypoint folder exists:", KEYPOINT_DIR.exists())
print("Total .npy files:", len(npy_files))

if npy_files:
    x = np.load(npy_files[0])

    print("Sample file:", npy_files[0].name)
    print("Sample shape:", x.shape)
    print("Sample dtype:", x.dtype)

Keypoint folder exists: True
Total .npy files: 5010
Sample file: 4463.npy
Sample shape: (32, 258)
Sample dtype: float32


#Keypoint copy

In [7]:
from pathlib import Path
import shutil

DRIVE_KEYPOINT_DIR = Path("/content/drive/MyDrive/CSE499B/keypoints_full")
LOCAL_KEYPOINT_DIR = Path("/content/keypoints_full")

# পুরোনো local copy থাকলে delete
if LOCAL_KEYPOINT_DIR.exists():
    shutil.rmtree(LOCAL_KEYPOINT_DIR)

# Drive -> Colab local storage
shutil.copytree(
    DRIVE_KEYPOINT_DIR,
    LOCAL_KEYPOINT_DIR
)

# Verify
local_files = list(
    LOCAL_KEYPOINT_DIR.rglob("*.npy")
)

print("Local folder exists:", LOCAL_KEYPOINT_DIR.exists())
print("Copied .npy files:", len(local_files))

if local_files:
    print("Example:", local_files[0])


Local folder exists: True
Copied .npy files: 5010
Example: /content/keypoints_full/3976.npy


In [8]:
import pandas as pd
from pathlib import Path

SPLIT_DIR = Path("/content/drive/MyDrive/CSE499B/split_v4_no_leak")

train_df = pd.read_csv(SPLIT_DIR / "train_manifest.csv")
val_df   = pd.read_csv(SPLIT_DIR / "val_manifest.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test_manifest.csv")

print("========== DATASET SIZE ==========")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

print("\n========== COLUMNS ==========")
print("Train columns:", train_df.columns.tolist())
print("Val columns:", val_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

print("\n========== TRAIN SAMPLE ==========")
print(train_df.head())

print("\n========== VAL SAMPLE ==========")
print(val_df.head())

print("\n========== TEST SAMPLE ==========")
print(test_df.head())

========== DATASET SIZE ==========
Train: 3506
Validation: 752
Test: 752
Total: 5010

========== COLUMNS ==========
Train columns: ['index', 'sentence', 'video_path']
Val columns: ['index', 'sentence', 'video_path']
Test columns: ['index', 'sentence', 'video_path']

========== TRAIN SAMPLE ==========
   index                 sentence                  video_path
0   2946               আমি অফিসার  data/videos/video/2946.mp4
1   2829   আমার মামা আফ্রিকা যাবে  data/videos/video/2829.mp4
2   2941  আমি অক্টোবরে নেপাল যাবো  data/videos/video/2941.mp4
3   3841          সে আমার ম্যাডাম  data/videos/video/3841.mp4
4   5404          সে আমার ম্যাডাম  data/videos/video/5404.mp4

========== VAL SAMPLE ==========
   index                               sentence                  video_path
0   3352  পাঞ্জাবি পরলে তোমাকে সুন্দর দেখা যায়  data/videos/video/3352.mp4
1   4819  পাঞ্জাবি পরলে তোমাকে সুন্দর দেখা যায়  data/videos/video/4819.mp4
2   5006                          আমার অনেক রাগ  data/videos/vid

#Data integrity + Leakage check

In [9]:
from pathlib import Path

LOCAL_KEYPOINT_DIR = Path("/content/keypoints_full")


def clean_id(x):
    x = str(x).strip()

    if x.endswith(".0"):
        x = x[:-2]

    return x


# -----------------------------
# 1. Available keypoint IDs
# -----------------------------

keypoint_ids = {
    p.stem
    for p in LOCAL_KEYPOINT_DIR.rglob("*.npy")
}


# -----------------------------
# 2. Manifest IDs
# -----------------------------

train_ids = {
    clean_id(x)
    for x in train_df["index"]
}

val_ids = {
    clean_id(x)
    for x in val_df["index"]
}

test_ids = {
    clean_id(x)
    for x in test_df["index"]
}


all_manifest_ids = (
    train_ids |
    val_ids |
    test_ids
)


# -----------------------------
# 3. Missing keypoints
# -----------------------------

missing_ids = (
    all_manifest_ids
    -
    keypoint_ids
)


# -----------------------------
# 4. Index overlap
# -----------------------------

train_val_id_overlap = train_ids & val_ids
train_test_id_overlap = train_ids & test_ids
val_test_id_overlap = val_ids & test_ids


# -----------------------------
# 5. Sentence overlap
# -----------------------------

train_sentences = set(
    train_df["sentence"].astype(str).str.strip()
)

val_sentences = set(
    val_df["sentence"].astype(str).str.strip()
)

test_sentences = set(
    test_df["sentence"].astype(str).str.strip()
)


train_val_sentence_overlap = (
    train_sentences & val_sentences
)

train_test_sentence_overlap = (
    train_sentences & test_sentences
)

val_test_sentence_overlap = (
    val_sentences & test_sentences
)


# -----------------------------
# Results
# -----------------------------

print("========== KEYPOINT MATCH ==========")

print(
    "Keypoint files:",
    len(keypoint_ids)
)

print(
    "Unique manifest IDs:",
    len(all_manifest_ids)
)

print(
    "Missing keypoints:",
    len(missing_ids)
)


print("\n========== INDEX LEAKAGE ==========")

print(
    "Train-Val ID overlap:",
    len(train_val_id_overlap)
)

print(
    "Train-Test ID overlap:",
    len(train_test_id_overlap)
)

print(
    "Val-Test ID overlap:",
    len(val_test_id_overlap)
)


print("\n========== SENTENCE LEAKAGE ==========")

print(
    "Train-Val sentence overlap:",
    len(train_val_sentence_overlap)
)

print(
    "Train-Test sentence overlap:",
    len(train_test_sentence_overlap)
)

print(
    "Val-Test sentence overlap:",
    len(val_test_sentence_overlap)
)


if missing_ids:
    print(
        "\nExample missing IDs:",
        list(missing_ids)[:20]
    )

========== KEYPOINT MATCH ==========
Keypoint files: 5010
Unique manifest IDs: 5010
Missing keypoints: 0

========== INDEX LEAKAGE ==========
Train-Val ID overlap: 0
Train-Test ID overlap: 0
Val-Test ID overlap: 0

========== SENTENCE LEAKAGE ==========
Train-Val sentence overlap: 0
Train-Test sentence overlap: 0
Val-Test sentence overlap: 0


#OOV check

In [10]:
import re
import unicodedata

def normalize_text(text):
    text = unicodedata.normalize("NFC", str(text))
    text = (
        text.replace("\u200c", "")
            .replace("\u200d", "")
            .replace("\ufeff", "")
    )

    text = re.sub(r"[^\w\u0980-\u09FF]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def tokenize(text):
    return normalize_text(text).split()


train_vocab = {
    w
    for s in train_df["sentence"]
    for w in tokenize(s)
}

val_vocab = {
    w
    for s in val_df["sentence"]
    for w in tokenize(s)
}

test_vocab = {
    w
    for s in test_df["sentence"]
    for w in tokenize(s)
}


val_oov = val_vocab - train_vocab
test_oov = test_vocab - train_vocab


print("========== VOCAB CHECK ==========")
print("Train vocabulary:", len(train_vocab))
print("Validation vocabulary:", len(val_vocab))
print("Test vocabulary:", len(test_vocab))

print("\nValidation OOV words:", len(val_oov))
print("Test OOV words:", len(test_oov))

print("\nExample validation OOV:")
print(list(sorted(val_oov))[:20])

print("\nExample test OOV:")
print(list(sorted(test_oov))[:20])

========== VOCAB CHECK ==========
Train vocabulary: 1154
Validation vocabulary: 554
Test vocabulary: 564

Validation OOV words: 128
Test OOV words: 130

Example validation OOV:
['F', 'অই', 'অভ্যন্তনা', 'অমর', 'অ্যামি', 'আখন', 'আগ্রহ', 'উথি', 'এম', 'এসেছো', 'ওই', 'করোনাভাইরাসেডাক্তারনার্সপুলিশ', 'কাজে', 'কামরাঙ্গা', 'কাল্কে', 'কোথা', 'কোনটা', 'কোরি', 'ক্লাস', 'খাদ্য']

Example test OOV:
['A', 'C', 'অইখানে', 'অপারানহু', 'অল্প', 'আকাশের', 'আগস্টের', 'আগামী', 'আবির', 'আব্বু', 'আমড়া', 'আরাম', 'আসবো', 'ইগো', 'ইশারার', 'উকিলের', 'উঠো', 'কয়েক', 'কর', 'করুন']


In [11]:
import pandas as pd
import re
import unicodedata

# --------------------------------------------------
# 1. Combine all 5010 samples
# --------------------------------------------------

all_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

all_df = all_df.drop_duplicates(
    subset=["index"]
).reset_index(drop=True)


# --------------------------------------------------
# 2. Conservative Bangla-safe normalization
# --------------------------------------------------

def normalize_bn(text):

    text = unicodedata.normalize(
        "NFC",
        str(text)
    )

    # Remove invisible unicode characters
    text = (
        text.replace("\u200c", "")
            .replace("\u200d", "")
            .replace("\ufeff", "")
    )

    # Replace punctuation with spaces
    text = re.sub(
        r'[!"#$%&\'()*+,\-./:;<=>?@\[\]^_`{|}~।…“”‘’–—،؛؟]',
        " ",
        text
    )

    # Collapse repeated whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


all_df["sentence_original"] = all_df["sentence"]

all_df["sentence"] = (
    all_df["sentence"]
    .astype(str)
    .map(normalize_bn)
)


# --------------------------------------------------
# 3. Basic audit
# --------------------------------------------------

print("========== V5 CLEAN DATA ==========")

print("Total rows:", len(all_df))

print(
    "Unique indices:",
    all_df["index"].nunique()
)

print(
    "Unique sentences:",
    all_df["sentence"].nunique()
)

print(
    "Empty sentences:",
    (all_df["sentence"].str.len() == 0).sum()
)

print(
    "Duplicate indices:",
    all_df["index"].duplicated().sum()
)


print("\n========== SAMPLE ==========")

print(
    all_df[
        [
            "index",
            "sentence_original",
            "sentence"
        ]
    ].head(10)
)

========== V5 CLEAN DATA ==========
Total rows: 5010
Unique indices: 5010
Unique sentences: 3378
Empty sentences: 0
Duplicate indices: 0

========== SAMPLE ==========
   index        sentence_original                 sentence
0   2946               আমি অফিসার               আমি অফিসার
1   2829   আমার মামা আফ্রিকা যাবে   আমার মামা আফ্রিকা যাবে
2   2941  আমি অক্টোবরে নেপাল যাবো  আমি অক্টোবরে নেপাল যাবো
3   3841          সে আমার ম্যাডাম          সে আমার ম্যাডাম
4   5404          সে আমার ম্যাডাম          সে আমার ম্যাডাম
5   1019            আমার মেয়ে আছে           আমার মেয়ে আছে
6   2762      আমার ভাই ইউরোপ যাবে      আমার ভাই ইউরোপ যাবে
7   3221  আমি সোমবার আমেরিকা যাবো  আমি সোমবার আমেরিকা যাবো
8   3810  তোমার মাতৃভাষা বাংলা কি  তোমার মাতৃভাষা বাংলা কি
9   5393  তোমার মাতৃভাষা বাংলা কি  তোমার মাতৃভাষা বাংলা কি


In [12]:
from collections import Counter

# Count word frequency over all 5010 samples
word_freq = Counter()

for sentence in all_df["sentence"]:
    for word in sentence.split():
        word_freq[word] += 1

# Basic statistics
freq_1 = sum(1 for w, c in word_freq.items() if c == 1)
freq_2 = sum(1 for w, c in word_freq.items() if c == 2)
freq_3plus = sum(1 for w, c in word_freq.items() if c >= 3)

print("========== WORD FREQUENCY AUDIT ==========")
print("Total unique words:", len(word_freq))
print("Words appearing once:", freq_1)
print("Words appearing exactly twice:", freq_2)
print("Words appearing 3+ times:", freq_3plus)

print("\n========== 20 MOST COMMON WORDS ==========")
for word, count in word_freq.most_common(20):
    print(f"{word:20s} {count}")

print("\n========== EXAMPLE RARE WORDS ==========")
rare_words = sorted(
    [w for w, c in word_freq.items() if c == 1]
)

print(rare_words[:50])

========== WORD FREQUENCY AUDIT ==========
Total unique words: 1398
Words appearing once: 153
Words appearing exactly twice: 500
Words appearing 3+ times: 745

========== 20 MOST COMMON WORDS ==========
আমার                 1996
আমি                  1166
যাবে                 573
যাব                  470
ভালো                 424
ভাই                  368
বোন                  315
তুমি                 299
খায়                 278
তোমার                260
অনেক                 227
লাগে                 223
বাবা                 218
মা                   200
করে                  192
কি                   185
যাবো                 176
পছন্দ                160
সুন্দর               143
খাবো                 140

========== EXAMPLE RARE WORDS ==========
['usa', 'অইখানে', 'আগামি', 'আগামী', 'আঙ্গুর', 'আনারশ', 'আপেল', 'আফ্রিকার', 'আবির', 'আমড়া', 'আমাই', 'আমারা', 'আমী', 'আরবি', 'আসবো', 'ইঞ্জিনিয়ার', 'উকিলের', 'উঠবো', 'এইখানে', 'এম', 'এসেছি', 'ওটার', 'কদবেল', 'কর', 'কাটি', 'কুকুরকে', 'কুলিকে', 'কেজি', 'কে

##Create a Small Vocabulary-Safe Demo Split

This step creates a small sentence-disjoint dataset for the first 20-epoch experiment.

Goals:
- Use approximately 1200 samples instead of all 5010 samples.
- Keep identical sentences in only one split to prevent leakage.
- Ensure that every word appearing in validation and test is already present in training.
- This small split is only for pipeline verification. The final experiment will later use the full dataset.

In [13]:
import random
import pandas as pd

RANDOM_SEED = 42

TARGET_TRAIN = 850
TARGET_VAL = 175
TARGET_TEST = 175

random.seed(RANDOM_SEED)


# --------------------------------------------------
# Group all samples by sentence
# --------------------------------------------------

sentence_groups = {
    sentence: group.copy()
    for sentence, group in all_df.groupby("sentence")
}

all_sentences = list(sentence_groups.keys())

random.shuffle(all_sentences)


# --------------------------------------------------
# STEP A: Build training groups first
# --------------------------------------------------

train_sentences_demo = []

train_count = 0

for sentence in all_sentences:

    train_sentences_demo.append(sentence)

    train_count += len(
        sentence_groups[sentence]
    )

    if train_count >= TARGET_TRAIN:
        break


train_sentence_set = set(
    train_sentences_demo
)


# --------------------------------------------------
# Build vocabulary only from selected training data
# --------------------------------------------------

train_demo_temp = pd.concat(
    [
        sentence_groups[s]
        for s in train_sentences_demo
    ],
    ignore_index=True
)


train_vocab_demo = {
    word
    for sentence in train_demo_temp["sentence"]
    for word in sentence.split()
}


# --------------------------------------------------
# STEP B: Find held-out sentences whose every word
# already exists in training vocabulary
# --------------------------------------------------

remaining_sentences = [
    s
    for s in all_sentences
    if s not in train_sentence_set
]


eligible_sentences = []

for sentence in remaining_sentences:

    words = set(
        sentence.split()
    )

    if words.issubset(
        train_vocab_demo
    ):
        eligible_sentences.append(sentence)


random.shuffle(
    eligible_sentences
)


# --------------------------------------------------
# STEP C: Build validation split
# --------------------------------------------------

val_sentences_demo = []

val_count = 0

for sentence in eligible_sentences:

    val_sentences_demo.append(
        sentence
    )

    val_count += len(
        sentence_groups[sentence]
    )

    if val_count >= TARGET_VAL:
        break


val_sentence_set = set(
    val_sentences_demo
)


# --------------------------------------------------
# STEP D: Build test split
# --------------------------------------------------

test_sentences_demo = []

test_count = 0

for sentence in eligible_sentences:

    if sentence in val_sentence_set:
        continue

    test_sentences_demo.append(
        sentence
    )

    test_count += len(
        sentence_groups[sentence]
    )

    if test_count >= TARGET_TEST:
        break


# --------------------------------------------------
# Create final DataFrames
# --------------------------------------------------

train_demo = pd.concat(
    [
        sentence_groups[s]
        for s in train_sentences_demo
    ],
    ignore_index=True
)


val_demo = pd.concat(
    [
        sentence_groups[s]
        for s in val_sentences_demo
    ],
    ignore_index=True
)


test_demo = pd.concat(
    [
        sentence_groups[s]
        for s in test_sentences_demo
    ],
    ignore_index=True
)


# --------------------------------------------------
# Final vocabulary check
# --------------------------------------------------

train_vocab_demo = {
    w
    for sentence in train_demo["sentence"]
    for w in sentence.split()
}

val_vocab_demo = {
    w
    for sentence in val_demo["sentence"]
    for w in sentence.split()
}

test_vocab_demo = {
    w
    for sentence in test_demo["sentence"]
    for w in sentence.split()
}


print("========== DEMO SPLIT SIZE ==========")

print("Train:", len(train_demo))
print("Validation:", len(val_demo))
print("Test:", len(test_demo))

print(
    "Total:",
    len(train_demo)
    + len(val_demo)
    + len(test_demo)
)


print("\n========== UNIQUE SENTENCES ==========")

print(
    "Train:",
    train_demo["sentence"].nunique()
)

print(
    "Validation:",
    val_demo["sentence"].nunique()
)

print(
    "Test:",
    test_demo["sentence"].nunique()
)


print("\n========== OOV CHECK ==========")

print(
    "Validation OOV:",
    len(
        val_vocab_demo
        -
        train_vocab_demo
    )
)

print(
    "Test OOV:",
    len(
        test_vocab_demo
        -
        train_vocab_demo
    )
)


print("\n========== SENTENCE LEAKAGE ==========")

print(
    "Train-Val overlap:",
    len(
        set(train_demo["sentence"])
        &
        set(val_demo["sentence"])
    )
)

print(
    "Train-Test overlap:",
    len(
        set(train_demo["sentence"])
        &
        set(test_demo["sentence"])
    )
)

print(
    "Val-Test overlap:",
    len(
        set(val_demo["sentence"])
        &
        set(test_demo["sentence"])
    )
)

========== DEMO SPLIT SIZE ==========
Train: 850
Validation: 175
Test: 175
Total: 1200

========== UNIQUE SENTENCES ==========
Train: 563
Validation: 129
Test: 126

========== OOV CHECK ==========
Validation OOV: 0
Test OOV: 0

========== SENTENCE LEAKAGE ==========
Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


##Build Separate CTC and Attention Vocabularies

The hybrid model uses the same Bangla training words but requires two different token-ID mappings.

### CTC vocabulary
CTC requires:
- `<blank>` token
- Bangla words

The blank token represents frames where no new word is emitted.

### Attention-decoder vocabulary
Following the Dhrubo SA-LSTM pipeline, the attention decoder requires:
- `<PAD>` for padding shorter target sentences
- `<SOS>` to start sentence generation
- `<EOS>` to stop sentence generation
- `<UNK>` for unknown words
- Bangla words

Both vocabularies are created only from the training split to avoid validation/test vocabulary leakage.

In [14]:
from collections import Counter

# --------------------------------------------------
# Collect words ONLY from demo training split
# --------------------------------------------------

train_word_counter = Counter()

for sentence in train_demo["sentence"]:
    train_word_counter.update(
        sentence.split()
    )

train_words = sorted(
    train_word_counter.keys()
)


# ==================================================
# 1. CTC VOCABULARY
# ==================================================

ctc_word_to_id = {
    "<blank>": 0
}

for word in train_words:
    ctc_word_to_id[word] = len(
        ctc_word_to_id
    )


ctc_id_to_word = {
    idx: word
    for word, idx
    in ctc_word_to_id.items()
}


# ==================================================
# 2. ATTENTION DECODER VOCABULARY
# ==================================================

attn_word_to_id = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "<UNK>": 3,
}

for word in train_words:
    attn_word_to_id[word] = len(
        attn_word_to_id
    )


attn_id_to_word = {
    idx: word
    for word, idx
    in attn_word_to_id.items()
}


# --------------------------------------------------
# Important IDs
# --------------------------------------------------

CTC_BLANK_ID = ctc_word_to_id["<blank>"]

PAD_ID = attn_word_to_id["<PAD>"]
SOS_ID = attn_word_to_id["<SOS>"]
EOS_ID = attn_word_to_id["<EOS>"]
UNK_ID = attn_word_to_id["<UNK>"]


# --------------------------------------------------
# Print summary
# --------------------------------------------------

print("========== TRAIN VOCABULARY ==========")

print(
    "Unique Bangla training words:",
    len(train_words)
)


print("\n========== CTC VOCAB ==========")

print(
    "CTC vocabulary size:",
    len(ctc_word_to_id)
)

print(
    "CTC blank ID:",
    CTC_BLANK_ID
)


print("\n========== ATTENTION VOCAB ==========")

print(
    "Attention vocabulary size:",
    len(attn_word_to_id)
)

print("PAD ID:", PAD_ID)
print("SOS ID:", SOS_ID)
print("EOS ID:", EOS_ID)
print("UNK ID:", UNK_ID)


print("\n========== SAMPLE WORD IDs ==========")

for word in train_words[:10]:

    print(
        word,
        "| CTC:",
        ctc_word_to_id[word],
        "| Attention:",
        attn_word_to_id[word]
    )

========== TRAIN VOCABULARY ==========
Unique Bangla training words: 584

========== CTC VOCAB ==========
CTC vocabulary size: 585
CTC blank ID: 0

========== ATTENTION VOCAB ==========
Attention vocabulary size: 588
PAD ID: 0
SOS ID: 1
EOS ID: 2
UNK ID: 3

========== SAMPLE WORD IDs ==========
C | CTC: 1 | Attention: 4
E | CTC: 2 | Attention: 5
অই | CTC: 3 | Attention: 6
অক্টোবর | CTC: 4 | Attention: 7
অক্টোবরে | CTC: 5 | Attention: 8
অদা | CTC: 6 | Attention: 9
অনেক | CTC: 7 | Attention: 10
অফিস | CTC: 8 | Attention: 11
অফিসে | CTC: 9 | Attention: 12
অমর | CTC: 10 | Attention: 13


##Check Target Lengths and CTC Feasibility

In [15]:
def ctc_required_steps(sentence):
    words = sentence.split()

    # Basic requirement = number of target words
    required = len(words)

    # Consecutive identical words need an extra blank timestep
    for i in range(1, len(words)):
        if words[i] == words[i - 1]:
            required += 1

    return required


def audit_split(df, name):

    word_lengths = df["sentence"].apply(
        lambda s: len(s.split())
    )

    ctc_steps = df["sentence"].apply(
        ctc_required_steps
    )

    impossible = ctc_steps > 32

    print(f"\n========== {name} ==========")

    print(
        "Samples:",
        len(df)
    )

    print(
        "Average target words:",
        round(word_lengths.mean(), 2)
    )

    print(
        "Maximum target words:",
        word_lengths.max()
    )

    print(
        "Maximum CTC required steps:",
        ctc_steps.max()
    )

    print(
        "CTC impossible samples:",
        impossible.sum()
    )

    if impossible.any():

        print("\nProblem examples:")

        print(
            df.loc[
                impossible,
                ["index", "sentence"]
            ].head(10)
        )

    return word_lengths.max()


train_max = audit_split(
    train_demo,
    "TRAIN"
)

val_max = audit_split(
    val_demo,
    "VALIDATION"
)

test_max = audit_split(
    test_demo,
    "TEST"
)


max_target_words = max(
    train_max,
    val_max,
    test_max
)


ATTENTION_MAX_LENGTH = (
    max_target_words + 2
)


print(
    "\n========== ATTENTION DECODER =========="
)

print(
    "Maximum sentence words:",
    max_target_words
)

print(
    "Maximum decoder length "
    "(including SOS + EOS):",
    ATTENTION_MAX_LENGTH
)


========== TRAIN ==========
Samples: 850
Average target words: 4.24
Maximum target words: 9
Maximum CTC required steps: 9
CTC impossible samples: 0

========== VALIDATION ==========
Samples: 175
Average target words: 4.2
Maximum target words: 8
Maximum CTC required steps: 8
CTC impossible samples: 0

========== TEST ==========
Samples: 175
Average target words: 4.06
Maximum target words: 7
Maximum CTC required steps: 7
CTC impossible samples: 0

========== ATTENTION DECODER ==========
Maximum sentence words: 9
Maximum decoder length (including SOS + EOS): 11


#Compute Training-Only Keypoint Normalization Statistics

In [16]:
import numpy as np
from pathlib import Path
from collections import Counter

LOCAL_KEYPOINT_DIR = Path("/content/keypoints_full")

FEATURE_DIM = 258


# --------------------------------------------------
# Build keypoint file map once
# --------------------------------------------------

keypoint_map = {
    p.stem: p
    for p in LOCAL_KEYPOINT_DIR.rglob("*.npy")
}


def clean_index(x):
    x = str(x).strip()

    if x.endswith(".0"):
        x = x[:-2]

    return x


# --------------------------------------------------
# Streaming statistics
# --------------------------------------------------

feature_sum = np.zeros(
    FEATURE_DIM,
    dtype=np.float64
)

feature_sq_sum = np.zeros(
    FEATURE_DIM,
    dtype=np.float64
)

total_frames = 0

shape_counter = Counter()

bad_shape_files = []
nonfinite_files = []
missing_files = []


for idx in train_demo["index"]:

    idx = clean_index(idx)

    if idx not in keypoint_map:
        missing_files.append(idx)
        continue

    path = keypoint_map[idx]

    x = np.load(path)

    shape_counter[x.shape] += 1


    # Expected [T, 258]
    if (
        x.ndim != 2
        or x.shape[1] != FEATURE_DIM
    ):
        bad_shape_files.append(
            (idx, x.shape)
        )
        continue


    # Check NaN / Inf
    if not np.isfinite(x).all():

        nonfinite_files.append(idx)

        x = np.nan_to_num(
            x,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )


    x64 = x.astype(
        np.float64
    )


    feature_sum += x64.sum(
        axis=0
    )

    feature_sq_sum += (
        x64 ** 2
    ).sum(
        axis=0
    )

    total_frames += x.shape[0]


# --------------------------------------------------
# Mean and standard deviation
# --------------------------------------------------

TRAIN_MEAN = (
    feature_sum
    /
    total_frames
)

variance = (
    feature_sq_sum
    /
    total_frames
) - (TRAIN_MEAN ** 2)

variance = np.maximum(
    variance,
    0.0
)

TRAIN_STD = np.sqrt(
    variance
)

# Avoid division by very tiny standard deviation
TRAIN_STD[
    TRAIN_STD < 1e-6
] = 1.0


# --------------------------------------------------
# Results
# --------------------------------------------------

print(
    "========== TRAINING KEYPOINT AUDIT =========="
)

print(
    "Training samples:",
    len(train_demo)
)

print(
    "Total training frames:",
    total_frames
)

print(
    "Observed shapes:",
    shape_counter
)

print(
    "Missing files:",
    len(missing_files)
)

print(
    "Bad shape files:",
    len(bad_shape_files)
)

print(
    "NaN/Inf files:",
    len(nonfinite_files)
)


print(
    "\n========== NORMALIZATION STATS =========="
)

print(
    "Feature dimension:",
    len(TRAIN_MEAN)
)

print(
    "Mean range:",
    float(TRAIN_MEAN.min()),
    "to",
    float(TRAIN_MEAN.max())
)

print(
    "Std range:",
    float(TRAIN_STD.min()),
    "to",
    float(TRAIN_STD.max())
)


print(
    "\nFirst 10 means:"
)

print(
    TRAIN_MEAN[:10]
)


print(
    "\nFirst 10 stds:"
)

print(
    TRAIN_STD[:10]
)

========== TRAINING KEYPOINT AUDIT ==========
Training samples: 850
Total training frames: 27200
Observed shapes: Counter({(32, 258): 850})
Missing files: 0
Bad shape files: 0
NaN/Inf files: 0

========== NORMALIZATION STATS ==========
Feature dimension: 258
Mean range: -1.72816143264065 to 1.6793836619829252
Std range: 0.014712164805894243 to 1.1385437853674918

First 10 means:
[ 0.48125658  0.39663706 -1.34358383  0.99804     0.5264171   0.38127941
 -1.25220055  0.9977243   0.5443627   0.39157265]

First 10 stds:
[0.07762074 0.06406712 0.64116683 0.02563813 0.07183134 0.07596422
 0.60226208 0.02542925 0.06390857 0.08344922]


#Build the Hybrid CTC + Attention Dataset

In [17]:
import torch
from torch.utils.data import Dataset
import numpy as np


# --------------------------------------------------
# Convert normalization statistics to float32
# --------------------------------------------------

TRAIN_MEAN_F32 = TRAIN_MEAN.astype(np.float32)
TRAIN_STD_F32 = TRAIN_STD.astype(np.float32)


# --------------------------------------------------
# Mild training-only augmentation
# --------------------------------------------------

def augment_keypoints(x):

    x = x.copy()

    # Small Gaussian feature noise
    if np.random.rand() < 0.40:
        noise = np.random.normal(
            loc=0.0,
            scale=0.01,
            size=x.shape
        ).astype(np.float32)

        x = x + noise


    # Randomly mask 1-2 timesteps
    if np.random.rand() < 0.30:

        mask_length = np.random.randint(
            1,
            3
        )

        start = np.random.randint(
            0,
            x.shape[0] - mask_length + 1
        )

        x[
            start:start + mask_length
        ] = 0.0


    return x


# --------------------------------------------------
# Hybrid Dataset
# --------------------------------------------------

class HybridSignDataset(Dataset):

    def __init__(
        self,
        dataframe,
        training=False
    ):

        self.df = dataframe.reset_index(
            drop=True
        ).copy()

        self.training = training


    def __len__(self):

        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        sample_id = clean_index(
            row["index"]
        )

        sentence = row["sentence"]

        words = sentence.split()


        # ------------------------------------------
        # Load keypoint features
        # ------------------------------------------

        if sample_id not in keypoint_map:

            raise FileNotFoundError(
                f"Missing keypoint file for ID: "
                f"{sample_id}"
            )


        x = np.load(
            keypoint_map[sample_id]
        ).astype(np.float32)


        if x.ndim != 2 or x.shape[1] != 258:

            raise ValueError(
                f"Bad keypoint shape for "
                f"{sample_id}: {x.shape}"
            )


        # ------------------------------------------
        # Normalize using TRAIN statistics only
        # ------------------------------------------

        x = (
            x - TRAIN_MEAN_F32
        ) / TRAIN_STD_F32


        # ------------------------------------------
        # Training-only augmentation
        # ------------------------------------------

        if self.training:

            x = augment_keypoints(x)


        # ------------------------------------------
        # CTC target
        # ------------------------------------------

        ctc_ids = []

        for word in words:

            if word not in ctc_word_to_id:

                raise ValueError(
                    f"CTC OOV word '{word}' "
                    f"in sample {sample_id}"
                )

            ctc_ids.append(
                ctc_word_to_id[word]
            )


        # ------------------------------------------
        # Attention target
        #
        # <SOS> sentence words <EOS>
        # ------------------------------------------

        attn_ids = [SOS_ID]

        for word in words:

            if word not in attn_word_to_id:

                raise ValueError(
                    f"Attention OOV word '{word}' "
                    f"in sample {sample_id}"
                )

            attn_ids.append(
                attn_word_to_id[word]
            )

        attn_ids.append(
            EOS_ID
        )


        # ------------------------------------------
        # Convert to PyTorch tensors
        # ------------------------------------------

        x = torch.tensor(
            x,
            dtype=torch.float32
        )

        ctc_target = torch.tensor(
            ctc_ids,
            dtype=torch.long
        )

        attn_target = torch.tensor(
            attn_ids,
            dtype=torch.long
        )


        return {
            "features": x,
            "ctc_target": ctc_target,
            "attn_target": attn_target,
            "sentence": sentence,
            "sample_id": sample_id
        }


# --------------------------------------------------
# Create datasets
# --------------------------------------------------

train_dataset = HybridSignDataset(
    train_demo,
    training=True
)

val_dataset = HybridSignDataset(
    val_demo,
    training=False
)

test_dataset = HybridSignDataset(
    test_demo,
    training=False
)


# --------------------------------------------------
# Test ONE validation sample
# No augmentation, so easier to inspect
# --------------------------------------------------

sample = val_dataset[0]


print("========== HYBRID DATASET TEST ==========")

print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)

print(
    "Test samples:",
    len(test_dataset)
)


print(
    "\nSample ID:",
    sample["sample_id"]
)

print(
    "Sentence:",
    sample["sentence"]
)

print(
    "Feature shape:",
    sample["features"].shape
)

print(
    "Feature dtype:",
    sample["features"].dtype
)


print(
    "\nCTC target IDs:",
    sample["ctc_target"].tolist()
)

print(
    "CTC target words:",
    [
        ctc_id_to_word[i]
        for i in sample[
            "ctc_target"
        ].tolist()
    ]
)


print(
    "\nAttention target IDs:",
    sample["attn_target"].tolist()
)

print(
    "Attention target words:",
    [
        attn_id_to_word[i]
        for i in sample[
            "attn_target"
        ].tolist()
    ]
)

========== HYBRID DATASET TEST ==========
Train samples: 850
Validation samples: 175
Test samples: 175

Sample ID: 5132
Sentence: তুমি আমার ভাই
Feature shape: torch.Size([32, 258])
Feature dtype: torch.float32

CTC target IDs: [225, 40, 406]
CTC target words: ['তুমি', 'আমার', 'ভাই']

Attention target IDs: [1, 228, 43, 409, 2]
Attention target words: ['<SOS>', 'তুমি', 'আমার', 'ভাই', '<EOS>']


#Build Hybrid Batch Collation and DataLoaders


This step combines individual samples into mini-batches for GPU training.

A batch must support both branches of the hybrid model:

### CTC branch
- Keypoint sequences are padded if necessary.
- Input lengths are preserved.
- CTC targets are concatenated into one 1D tensor.
- Individual target lengths are stored separately for `CTCLoss`.

### Attention branch
- Target sequences are padded using `<PAD>`.
- The padded sequence contains `<SOS>`, Bangla words, and `<EOS>`.
- During training, the decoder will later use:
  - all tokens except the last as decoder input
  - all tokens except the first as prediction targets

A batch size of 16 is used for the first 20-epoch demo experiment.

In [18]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch


# ==================================================
# Hybrid collate function
# ==================================================

def hybrid_collate(batch):

    # --------------------------------------------------
    # 1. FEATURES
    # --------------------------------------------------

    features = [
        item["features"]
        for item in batch
    ]

    input_lengths = torch.tensor(
        [
            x.shape[0]
            for x in features
        ],
        dtype=torch.long
    )

    # [B, Tmax, 258]
    features_padded = pad_sequence(
        features,
        batch_first=True,
        padding_value=0.0
    )


    # --------------------------------------------------
    # 2. CTC TARGETS
    # --------------------------------------------------

    ctc_targets_list = [
        item["ctc_target"]
        for item in batch
    ]

    ctc_target_lengths = torch.tensor(
        [
            len(target)
            for target in ctc_targets_list
        ],
        dtype=torch.long
    )

    # CTCLoss expects one concatenated target tensor
    ctc_targets = torch.cat(
        ctc_targets_list
    )


    # --------------------------------------------------
    # 3. ATTENTION TARGETS
    # --------------------------------------------------

    attn_targets_list = [
        item["attn_target"]
        for item in batch
    ]

    attn_target_lengths = torch.tensor(
        [
            len(target)
            for target in attn_targets_list
        ],
        dtype=torch.long
    )

    # [B, max_target_length]
    attn_targets = pad_sequence(
        attn_targets_list,
        batch_first=True,
        padding_value=PAD_ID
    )


    # --------------------------------------------------
    # 4. Metadata
    # --------------------------------------------------

    sentences = [
        item["sentence"]
        for item in batch
    ]

    sample_ids = [
        item["sample_id"]
        for item in batch
    ]


    return {
        "features": features_padded,

        "input_lengths": input_lengths,

        "ctc_targets": ctc_targets,

        "ctc_target_lengths": ctc_target_lengths,

        "attn_targets": attn_targets,

        "attn_target_lengths": attn_target_lengths,

        "sentences": sentences,

        "sample_ids": sample_ids
    }


# ==================================================
# DataLoaders
# ==================================================

BATCH_SIZE = 16


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=hybrid_collate,
    num_workers=2,
    pin_memory=True
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=hybrid_collate,
    num_workers=2,
    pin_memory=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=hybrid_collate,
    num_workers=2,
    pin_memory=True
)


# ==================================================
# Test one validation batch
# ==================================================

batch = next(
    iter(val_loader)
)


print(
    "========== HYBRID BATCH TEST =========="
)

print(
    "Features:",
    batch["features"].shape
)

print(
    "Input lengths:",
    batch["input_lengths"].shape
)

print(
    "CTC targets:",
    batch["ctc_targets"].shape
)

print(
    "CTC target lengths:",
    batch["ctc_target_lengths"].shape
)

print(
    "Attention targets:",
    batch["attn_targets"].shape
)

print(
    "Attention target lengths:",
    batch["attn_target_lengths"].shape
)


print(
    "\nFirst sample ID:",
    batch["sample_ids"][0]
)

print(
    "First sentence:",
    batch["sentences"][0]
)


print(
    "\nFirst attention target IDs:"
)

first_attn_len = int(
    batch[
        "attn_target_lengths"
    ][0]
)

print(
    batch[
        "attn_targets"
    ][0, :first_attn_len].tolist()
)


print(
    "First attention target words:"
)

print(
    [
        attn_id_to_word[i]
        for i in batch[
            "attn_targets"
        ][0, :first_attn_len].tolist()
    ]
)

========== HYBRID BATCH TEST ==========
Features: torch.Size([16, 32, 258])
Input lengths: torch.Size([16])
CTC targets: torch.Size([66])
CTC target lengths: torch.Size([16])
Attention targets: torch.Size([16, 8])
Attention target lengths: torch.Size([16])

First sample ID: 5132
First sentence: তুমি আমার ভাই

First attention target IDs:
[1, 228, 43, 409, 2]
First attention target words:
['<SOS>', 'তুমি', 'আমার', 'ভাই', '<EOS>']


#Build the V5 Hybrid CTC + Temporal-Attention Model


Architecture:

Keypoints `[B, 32, 258]`
→ Temporal CNN
→ Bidirectional LSTM Encoder
→ Shared encoded representation

 The encoded representation is sent to two branches:

1. **CTC branch**
   - Frame-wise word classification
   - CTC loss learns alignment without frame-level word labels

2. **Attention branch**
   - Temporal Attention selects important encoded timesteps
   - LSTM decoder generates the Bangla sentence word-by-word
   - Uses `<SOS>` and `<EOS>` tokens

The two losses will later be combined during training.

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)

# ==================================================
# Configuration
# ==================================================

INPUT_DIM = 258

CNN_DIM = 258

ENCODER_HIDDEN = 256
ENCODER_LAYERS = 2
ENCODER_DIM = ENCODER_HIDDEN * 2   # BiLSTM = 512

DECODER_HIDDEN = 512
EMBED_DIM = 256

DROPOUT = 0.40


# ==================================================
# 1. Temporal Residual CNN Block
# ==================================================

class TemporalResidualBlock(nn.Module):

    def __init__(self, dim, dropout=0.4):
        super().__init__()

        self.conv1 = nn.Conv1d(
            dim,
            dim,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv1d(
            dim,
            dim,
            kernel_size=3,
            padding=1
        )

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

        self.dropout = nn.Dropout(dropout)


    def forward(self, x):

        # x = [B, T, F]

        residual = x

        y = x.transpose(1, 2)

        y = self.conv1(y)

        y = y.transpose(1, 2)

        y = self.norm1(y)

        y = F.relu(y)

        y = self.dropout(y)


        y = y.transpose(1, 2)

        y = self.conv2(y)

        y = y.transpose(1, 2)

        y = self.norm2(y)

        y = self.dropout(y)

        return F.relu(
            residual + y
        )


# ==================================================
# 2. Shared CNN + BiLSTM Encoder
# ==================================================

class SignEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.cnn = nn.Sequential(
            TemporalResidualBlock(
                CNN_DIM,
                DROPOUT
            ),
            TemporalResidualBlock(
                CNN_DIM,
                DROPOUT
            ),
            TemporalResidualBlock(
                CNN_DIM,
                DROPOUT
            )
        )


        self.bilstm = nn.LSTM(
            input_size=CNN_DIM,
            hidden_size=ENCODER_HIDDEN,
            num_layers=ENCODER_LAYERS,
            batch_first=True,
            bidirectional=True,
            dropout=DROPOUT
        )


    def forward(
        self,
        x,
        lengths
    ):

        # CNN
        x = self.cnn(x)

        # Ignore padded frames inside BiLSTM
        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, _ = self.bilstm(
            packed
        )

        encoded, _ = pad_packed_sequence(
            packed_out,
            batch_first=True
        )

        # encoded = [B, T, 512]

        return encoded


# ==================================================
# 3. Dhrubo-inspired Temporal Attention
# ==================================================

class TemporalAttention(nn.Module):

    def __init__(
        self,
        encoder_dim,
        decoder_dim,
        attention_dim=256
    ):

        super().__init__()

        self.encoder_proj = nn.Linear(
            encoder_dim,
            attention_dim
        )

        self.decoder_proj = nn.Linear(
            decoder_dim,
            attention_dim
        )

        self.energy = nn.Linear(
            attention_dim,
            1
        )


    def forward(
        self,
        encoder_outputs,
        decoder_hidden,
        mask=None
    ):

        # encoder_outputs = [B,T,512]
        # decoder_hidden  = [B,512]

        enc = self.encoder_proj(
            encoder_outputs
        )

        dec = self.decoder_proj(
            decoder_hidden
        ).unsqueeze(1)

        scores = self.energy(
            torch.tanh(
                enc + dec
            )
        ).squeeze(-1)


        if mask is not None:

            scores = scores.masked_fill(
                ~mask,
                -1e9
            )


        attention_weights = F.softmax(
            scores,
            dim=1
        )

        context = torch.bmm(
            attention_weights.unsqueeze(1),
            encoder_outputs
        ).squeeze(1)

        return context, attention_weights


# ==================================================
# 4. Attention LSTM Decoder
# ==================================================

class AttentionDecoder(nn.Module):

    def __init__(
        self,
        vocab_size
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            EMBED_DIM,
            padding_idx=PAD_ID
        )


        self.attention = TemporalAttention(
            ENCODER_DIM,
            DECODER_HIDDEN
        )


        self.lstm = nn.LSTMCell(
            EMBED_DIM + ENCODER_DIM,
            DECODER_HIDDEN
        )


        self.init_h = nn.Linear(
            ENCODER_DIM,
            DECODER_HIDDEN
        )

        self.init_c = nn.Linear(
            ENCODER_DIM,
            DECODER_HIDDEN
        )


        self.output = nn.Linear(
            DECODER_HIDDEN + ENCODER_DIM,
            vocab_size
        )


        self.dropout = nn.Dropout(
            DROPOUT
        )


    def forward(
        self,
        encoder_outputs,
        lengths,
        decoder_inputs
    ):

        B, T, _ = encoder_outputs.shape

        device = encoder_outputs.device


        # ------------------------------------------
        # Padding mask
        # ------------------------------------------

        mask = (
            torch.arange(
                T,
                device=device
            ).unsqueeze(0)
            <
            lengths.to(device).unsqueeze(1)
        )


        # ------------------------------------------
        # Mean encoder representation
        # for decoder initialization
        # ------------------------------------------

        mask_float = (
            mask.unsqueeze(-1).float()
        )

        mean_encoded = (
            encoder_outputs
            *
            mask_float
        ).sum(dim=1)

        mean_encoded = (
            mean_encoded
            /
            lengths.to(device)
            .unsqueeze(1)
            .float()
        )


        h = torch.tanh(
            self.init_h(mean_encoded)
        )

        c = torch.tanh(
            self.init_c(mean_encoded)
        )


        logits_list = []


        # decoder_inputs:
        # [SOS, word1, word2, ...]

        for t in range(
            decoder_inputs.size(1)
        ):

            token = decoder_inputs[:, t]

            emb = self.embedding(
                token
            )


            context, _ = self.attention(
                encoder_outputs,
                h,
                mask
            )


            lstm_input = torch.cat(
                [
                    emb,
                    context
                ],
                dim=1
            )


            h, c = self.lstm(
                lstm_input,
                (h, c)
            )


            output_input = torch.cat(
                [
                    self.dropout(h),
                    context
                ],
                dim=1
            )


            logits = self.output(
                output_input
            )


            logits_list.append(
                logits.unsqueeze(1)
            )


        return torch.cat(
            logits_list,
            dim=1
        )


# ==================================================
# 5. Complete Hybrid Model
# ==================================================

class HybridCTCAttentionModel(nn.Module):

    def __init__(
        self,
        ctc_vocab_size,
        attn_vocab_size
    ):

        super().__init__()


        # Shared Encoder
        self.encoder = SignEncoder()


        # CTC branch
        self.ctc_head = nn.Linear(
            ENCODER_DIM,
            ctc_vocab_size
        )


        # Attention branch
        self.decoder = AttentionDecoder(
            attn_vocab_size
        )


    def forward(
        self,
        features,
        input_lengths,
        decoder_inputs
    ):

        # Shared encoder
        encoded = self.encoder(
            features,
            input_lengths
        )


        # -------------------------
        # CTC branch
        # -------------------------

        ctc_logits = self.ctc_head(
            encoded
        )

        ctc_log_probs = F.log_softmax(
            ctc_logits,
            dim=-1
        )

        # CTCLoss expects [T,B,C]
        ctc_log_probs = (
            ctc_log_probs
            .transpose(0, 1)
        )


        # -------------------------
        # Attention branch
        # -------------------------

        attn_logits = self.decoder(
            encoded,
            input_lengths,
            decoder_inputs
        )


        return (
            ctc_log_probs,
            attn_logits
        )


# ==================================================
# 6. Create Model
# ==================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


model = HybridCTCAttentionModel(
    ctc_vocab_size=len(
        ctc_word_to_id
    ),
    attn_vocab_size=len(
        attn_word_to_id
    )
).to(device)


# ==================================================
# 7. Test one batch
# ==================================================

batch = next(
    iter(val_loader)
)


features = batch[
    "features"
].to(device)

input_lengths = batch[
    "input_lengths"
]


attn_targets = batch[
    "attn_targets"
].to(device)


# Decoder input excludes final token
decoder_inputs = (
    attn_targets[:, :-1]
)


with torch.no_grad():

    ctc_log_probs, attn_logits = model(
        features,
        input_lengths,
        decoder_inputs
    )


print(
    "========== V5 MODEL TEST =========="
)

print(
    "Device:",
    device
)

print(
    "Input:",
    features.shape
)

print(
    "Encoder expected:",
    "[B, 32, 512]"
)

print(
    "CTC output:",
    ctc_log_probs.shape
)

print(
    "Attention output:",
    attn_logits.shape
)

print(
    "CTC vocab:",
    len(ctc_word_to_id)
)

print(
    "Attention vocab:",
    len(attn_word_to_id)
)

print(
    "Total parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

========== V5 MODEL TEST ==========
Device: cuda
Input: torch.Size([16, 32, 258])
Encoder expected: [B, 32, 512]
CTC output: torch.Size([32, 16, 585])
Attention output: torch.Size([16, 7, 588])
CTC vocab: 585
Attention vocab: 588
Total parameters: 8303618


#Train V5 Hybrid CTC + Temporal-Attention Model for 20 Epochs

In [20]:
import os
import math
import pandas as pd
import torch
import torch.nn as nn

# =========================================================
# CONFIG
# =========================================================

EPOCHS = 20
CTC_WEIGHT = 0.30
ATTN_WEIGHT = 0.70

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

SAVE_DIR = Path(
    "/content/drive/MyDrive/CSE499B/v5_hybrid_demo"
)
SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_MODEL_PATH = SAVE_DIR / "best_v5_hybrid.pt"


# =========================================================
# LOSSES
# =========================================================

ctc_criterion = nn.CTCLoss(
    blank=CTC_BLANK_ID,
    zero_infinity=True
)

attn_criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)


# =========================================================
# OPTIMIZER
# =========================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)


# =========================================================
# WORD ERROR RATE
# =========================================================

def edit_distance(ref, hyp):

    n = len(ref)
    m = len(hyp)

    dp = [
        [0] * (m + 1)
        for _ in range(n + 1)
    ]

    for i in range(n + 1):
        dp[i][0] = i

    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):

        for j in range(1, m + 1):

            if ref[i - 1] == hyp[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]

            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],
                    dp[i][j - 1],
                    dp[i - 1][j - 1]
                )

    return dp[n][m]


# =========================================================
# AUTOREGRESSIVE ATTENTION DECODER
# =========================================================

@torch.no_grad()
def attention_greedy_decode(
    model,
    features,
    lengths,
    max_steps=10
):

    model.eval()

    encoder_outputs = model.encoder(
        features,
        lengths
    )

    B, T, _ = encoder_outputs.shape

    device_now = encoder_outputs.device


    # -----------------------------
    # Encoder mask
    # -----------------------------

    mask = (
        torch.arange(
            T,
            device=device_now
        ).unsqueeze(0)
        <
        lengths.to(device_now).unsqueeze(1)
    )


    # -----------------------------
    # Initial decoder state
    # -----------------------------

    mask_float = mask.unsqueeze(-1).float()

    mean_encoded = (
        encoder_outputs *
        mask_float
    ).sum(dim=1)

    mean_encoded = (
        mean_encoded
        /
        lengths.to(device_now)
        .unsqueeze(1)
        .float()
    )


    decoder = model.decoder

    h = torch.tanh(
        decoder.init_h(mean_encoded)
    )

    c = torch.tanh(
        decoder.init_c(mean_encoded)
    )


    current_token = torch.full(
        (B,),
        SOS_ID,
        dtype=torch.long,
        device=device_now
    )


    predictions = [
        []
        for _ in range(B)
    ]

    finished = torch.zeros(
        B,
        dtype=torch.bool,
        device=device_now
    )


    # -----------------------------
    # Generate one word at a time
    # -----------------------------

    for _ in range(max_steps):

        embedding = decoder.embedding(
            current_token
        )

        context, _ = decoder.attention(
            encoder_outputs,
            h,
            mask
        )

        decoder_input = torch.cat(
            [
                embedding,
                context
            ],
            dim=1
        )

        h, c = decoder.lstm(
            decoder_input,
            (h, c)
        )

        logits = decoder.output(
            torch.cat(
                [
                    h,
                    context
                ],
                dim=1
            )
        )

        next_token = logits.argmax(
            dim=-1
        )


        for i in range(B):

            if finished[i]:
                continue

            token_id = int(
                next_token[i].item()
            )

            if token_id == EOS_ID:

                finished[i] = True

            else:

                predictions[i].append(
                    token_id
                )


        current_token = next_token


        if finished.all():
            break


    # -----------------------------
    # IDs -> words
    # -----------------------------

    decoded_words = []

    for sequence in predictions:

        words = []

        for token_id in sequence:

            word = attn_id_to_word[
                token_id
            ]

            if word in {
                "<PAD>",
                "<SOS>",
                "<EOS>"
            }:
                continue

            words.append(word)

        decoded_words.append(words)


    return decoded_words


# =========================================================
# VALIDATION / TEST FUNCTION
# =========================================================

@torch.no_grad()
def evaluate_hybrid(loader):

    model.eval()

    total_loss = 0.0
    total_ctc_loss = 0.0
    total_attn_loss = 0.0

    total_samples = 0

    total_edit_distance = 0
    total_reference_words = 0

    exact_correct = 0


    for batch in loader:

        features = batch[
            "features"
        ].to(device)

        input_lengths = batch[
            "input_lengths"
        ]

        ctc_targets = batch[
            "ctc_targets"
        ].to(device)

        ctc_target_lengths = batch[
            "ctc_target_lengths"
        ]

        attn_targets = batch[
            "attn_targets"
        ].to(device)


        # -----------------------------------------
        # Teacher-forced attention training format
        # -----------------------------------------

        decoder_inputs = (
            attn_targets[:, :-1]
        )

        decoder_targets = (
            attn_targets[:, 1:]
        )


        # -----------------------------------------
        # Forward
        # -----------------------------------------

        ctc_log_probs, attn_logits = model(
            features,
            input_lengths,
            decoder_inputs
        )


        # -----------------------------------------
        # CTC loss
        # -----------------------------------------

        ctc_loss = ctc_criterion(
            ctc_log_probs,
            ctc_targets,
            input_lengths,
            ctc_target_lengths
        )


        # -----------------------------------------
        # Attention loss
        # -----------------------------------------

        attn_loss = attn_criterion(
            attn_logits.reshape(
                -1,
                attn_logits.size(-1)
            ),
            decoder_targets.reshape(-1)
        )


        loss = (
            CTC_WEIGHT * ctc_loss
            +
            ATTN_WEIGHT * attn_loss
        )


        batch_size_now = features.size(0)

        total_loss += (
            loss.item() *
            batch_size_now
        )

        total_ctc_loss += (
            ctc_loss.item() *
            batch_size_now
        )

        total_attn_loss += (
            attn_loss.item() *
            batch_size_now
        )

        total_samples += batch_size_now


        # -----------------------------------------
        # Real autoregressive prediction
        # -----------------------------------------

        predictions = attention_greedy_decode(
            model,
            features,
            input_lengths,
            max_steps=ATTENTION_MAX_LENGTH - 1
        )


        # -----------------------------------------
        # WER + exact sentence accuracy
        # -----------------------------------------

        for reference_sentence, predicted_words in zip(
            batch["sentences"],
            predictions
        ):

            reference_words = (
                reference_sentence.split()
            )

            total_edit_distance += edit_distance(
                reference_words,
                predicted_words
            )

            total_reference_words += len(
                reference_words
            )


            if (
                reference_words
                ==
                predicted_words
            ):
                exact_correct += 1


    wer = (
        total_edit_distance
        /
        max(
            total_reference_words,
            1
        )
    )

    exact_accuracy = (
        exact_correct
        /
        max(
            total_samples,
            1
        )
    )


    return {
        "loss":
            total_loss / total_samples,

        "ctc_loss":
            total_ctc_loss / total_samples,

        "attn_loss":
            total_attn_loss / total_samples,

        "wer":
            wer,

        "sentence_accuracy":
            exact_accuracy
    }


# =========================================================
# TRAINING
# =========================================================

best_val_wer = float("inf")

history = []


print(
    "=============================================="
)

print(
    "V5 HYBRID CTC + ATTENTION TRAINING"
)

print(
    "=============================================="
)

print(
    "Device:",
    device
)

print(
    "Epochs:",
    EPOCHS
)

print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)

print(
    "CTC weight:",
    CTC_WEIGHT
)

print(
    "Attention weight:",
    ATTN_WEIGHT
)

print()


for epoch in range(
    1,
    EPOCHS + 1
):

    model.train()

    running_loss = 0.0

    running_ctc = 0.0

    running_attn = 0.0

    seen_samples = 0


    for batch in train_loader:

        features = batch[
            "features"
        ].to(
            device,
            non_blocking=True
        )

        input_lengths = batch[
            "input_lengths"
        ]

        ctc_targets = batch[
            "ctc_targets"
        ].to(
            device,
            non_blocking=True
        )

        ctc_target_lengths = batch[
            "ctc_target_lengths"
        ]

        attn_targets = batch[
            "attn_targets"
        ].to(
            device,
            non_blocking=True
        )


        # Attention teacher forcing
        decoder_inputs = (
            attn_targets[:, :-1]
        )

        decoder_targets = (
            attn_targets[:, 1:]
        )


        optimizer.zero_grad()


        # -----------------------------------------
        # Forward
        # -----------------------------------------

        ctc_log_probs, attn_logits = model(
            features,
            input_lengths,
            decoder_inputs
        )


        # -----------------------------------------
        # Losses
        # -----------------------------------------

        ctc_loss = ctc_criterion(
            ctc_log_probs,
            ctc_targets,
            input_lengths,
            ctc_target_lengths
        )


        attn_loss = attn_criterion(
            attn_logits.reshape(
                -1,
                attn_logits.size(-1)
            ),
            decoder_targets.reshape(-1)
        )


        total_loss = (
            CTC_WEIGHT * ctc_loss
            +
            ATTN_WEIGHT * attn_loss
        )


        # -----------------------------------------
        # Backpropagation
        # -----------------------------------------

        total_loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )


        optimizer.step()


        batch_size_now = features.size(0)


        running_loss += (
            total_loss.item()
            *
            batch_size_now
        )

        running_ctc += (
            ctc_loss.item()
            *
            batch_size_now
        )

        running_attn += (
            attn_loss.item()
            *
            batch_size_now
        )

        seen_samples += batch_size_now


    train_loss = (
        running_loss /
        seen_samples
    )

    train_ctc = (
        running_ctc /
        seen_samples
    )

    train_attn = (
        running_attn /
        seen_samples
    )


    # =============================================
    # Validation
    # =============================================

    val_result = evaluate_hybrid(
        val_loader
    )


    scheduler.step(
        val_result["wer"]
    )


    current_lr = optimizer.param_groups[
        0
    ]["lr"]


    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train={train_loss:.4f} | "
        f"CTC={train_ctc:.4f} | "
        f"Attn={train_attn:.4f} | "
        f"Val={val_result['loss']:.4f} | "
        f"WER={val_result['wer']:.4f} | "
        f"Exact={val_result['sentence_accuracy']:.4f} | "
        f"LR={current_lr:.6f}"
    )


    history.append({
        "epoch": epoch,

        "train_loss":
            train_loss,

        "train_ctc_loss":
            train_ctc,

        "train_attn_loss":
            train_attn,

        "val_loss":
            val_result["loss"],

        "val_wer":
            val_result["wer"],

        "val_sentence_accuracy":
            val_result[
                "sentence_accuracy"
            ],

        "lr":
            current_lr
    })


    # =============================================
    # Save best model
    # =============================================

    if (
        val_result["wer"]
        <
        best_val_wer
    ):

        best_val_wer = (
            val_result["wer"]
        )


        torch.save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "best_val_wer":
                    best_val_wer,

                "ctc_word_to_id":
                    ctc_word_to_id,

                "attn_word_to_id":
                    attn_word_to_id,

                "train_mean":
                    TRAIN_MEAN_F32,

                "train_std":
                    TRAIN_STD_F32,

                "config": {
                    "input_dim":
                        INPUT_DIM,

                    "encoder_hidden":
                        ENCODER_HIDDEN,

                    "decoder_hidden":
                        DECODER_HIDDEN,

                    "embed_dim":
                        EMBED_DIM,

                    "ctc_weight":
                        CTC_WEIGHT,

                    "attn_weight":
                        ATTN_WEIGHT
                }
            },

            BEST_MODEL_PATH
        )


        print(
            f"   ✅ Best model saved "
            f"(Val WER={best_val_wer:.4f})"
        )


# =========================================================
# SAVE TRAINING HISTORY
# =========================================================

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    SAVE_DIR /
    "training_history.csv",
    index=False
)


# =========================================================
# LOAD BEST MODEL
# =========================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


# =========================================================
# FINAL TEST
# =========================================================

test_result = evaluate_hybrid(
    test_loader
)


print(
    "\n=============================================="
)

print(
    "FINAL DEMO RESULT"
)

print(
    "=============================================="
)

print(
    "Best Epoch:",
    checkpoint["epoch"]
)

print(
    "Best Validation WER:",
    round(
        checkpoint[
            "best_val_wer"
        ],
        4
    )
)

print(
    "Test Loss:",
    round(
        test_result["loss"],
        4
    )
)

print(
    "Test CTC Loss:",
    round(
        test_result["ctc_loss"],
        4
    )
)

print(
    "Test Attention Loss:",
    round(
        test_result["attn_loss"],
        4
    )
)

print(
    "Test WER:",
    round(
        test_result["wer"],
        4
    )
)

print(
    "Exact Sentence Accuracy:",
    round(
        test_result[
            "sentence_accuracy"
        ],
        4
    )
)

print(
    "\nBest model saved at:"
)

print(
    BEST_MODEL_PATH
)

V5 HYBRID CTC + ATTENTION TRAINING
Device: cuda
Epochs: 20
Train samples: 850
Validation samples: 175
CTC weight: 0.3
Attention weight: 0.7

Epoch 01/20 | Train=6.7679 | CTC=11.1923 | Attn=4.8717 | Val=4.3809 | WER=0.8218 | Exact=0.0000 | LR=0.000300
   ✅ Best model saved (Val WER=0.8218)
Epoch 02/20 | Train=4.6073 | CTC=5.8530 | Attn=4.0734 | Val=4.1384 | WER=0.8762 | Exact=0.0000 | LR=0.000300
Epoch 03/20 | Train=4.2225 | CTC=5.6181 | Attn=3.6243 | Val=3.8132 | WER=0.8721 | Exact=0.0000 | LR=0.000300
Epoch 04/20 | Train=3.8471 | CTC=5.3834 | Attn=3.1887 | Val=3.5862 | WER=0.9238 | Exact=0.0000 | LR=0.000300
Epoch 05/20 | Train=3.5153 | CTC=5.1380 | Attn=2.8198 | Val=3.4021 | WER=0.8993 | Exact=0.0000 | LR=0.000150
Epoch 06/20 | Train=3.2315 | CTC=4.9335 | Attn=2.5020 | Val=3.3006 | WER=0.9347 | Exact=0.0000 | LR=0.000150
Epoch 07/20 | Train=3.0898 | CTC=4.8318 | Attn=2.3432 | Val=3.2240 | WER=0.9306 | Exact=0.0000 | LR=0.000150
Epoch 08/20 | Train=2.9555 | CTC=4.7288 | Attn=2.1955 | 

##Create the Full V5 Sentence-Disjoint and Vocabulary-Safe Split

The initial 1200-sample experiment confirmed that the hybrid pipeline runs correctly, but the dataset was too small for reliable sequence generation.

We now return to all 5,010 samples.

This split:
- Uses all available samples.
- Keeps identical sentences in only one split.
- Targets approximately 70% training, 15% validation, and 15% test data.
- Ensures that every word appearing in validation and test also appears in training.
- Keeps rare-word sentence groups in training when necessary.

This full V5 split will be used for the main experiment.

In [22]:
import random
import pandas as pd
from collections import Counter

RANDOM_SEED = 42

TARGET_VAL = 752
TARGET_TEST = 752

random.seed(RANDOM_SEED)


# =========================================================
# Group samples by normalized sentence
# =========================================================

groups = {
    sentence: group.copy()
    for sentence, group
    in all_df.groupby("sentence")
}

sentences = list(groups.keys())

random.shuffle(sentences)


# =========================================================
# Count how many UNIQUE sentence groups contain each word
# =========================================================

word_group_count = Counter()

for sentence in sentences:

    for word in set(sentence.split()):

        word_group_count[word] += 1


# Initially every sentence group belongs to training
remaining_train_word_groups = dict(
    word_group_count
)


# =========================================================
# Create validation and test groups
# =========================================================

val_sentences_full = []
test_sentences_full = []

val_count = 0
test_count = 0


for sentence in sentences:

    words = set(
        sentence.split()
    )

    group_size = len(
        groups[sentence]
    )


    # If this group is removed from training,
    # every word must still remain in at least
    # one training sentence group.
    safe_to_holdout = all(
        remaining_train_word_groups[word] >= 2
        for word in words
    )


    if not safe_to_holdout:
        continue


    # Prefer whichever held-out split still needs more data
    val_need = TARGET_VAL - val_count
    test_need = TARGET_TEST - test_count


    if val_need <= 0 and test_need <= 0:
        break


    if val_need >= test_need and val_need > 0:

        val_sentences_full.append(
            sentence
        )

        val_count += group_size

    elif test_need > 0:

        test_sentences_full.append(
            sentence
        )

        test_count += group_size

    else:
        continue


    # This sentence group is no longer in training
    for word in words:

        remaining_train_word_groups[word] -= 1


# =========================================================
# Everything else stays in training
# =========================================================

heldout_sentences = (
    set(val_sentences_full)
    |
    set(test_sentences_full)
)

train_sentences_full = [
    sentence
    for sentence in sentences
    if sentence not in heldout_sentences
]


# =========================================================
# Build DataFrames
# =========================================================

train_full = pd.concat(
    [
        groups[s]
        for s in train_sentences_full
    ],
    ignore_index=True
)

val_full = pd.concat(
    [
        groups[s]
        for s in val_sentences_full
    ],
    ignore_index=True
)

test_full = pd.concat(
    [
        groups[s]
        for s in test_sentences_full
    ],
    ignore_index=True
)


# =========================================================
# Vocabulary audit
# =========================================================

train_vocab_full = {
    word
    for sentence in train_full["sentence"]
    for word in sentence.split()
}

val_vocab_full = {
    word
    for sentence in val_full["sentence"]
    for word in sentence.split()
}

test_vocab_full = {
    word
    for sentence in test_full["sentence"]
    for word in sentence.split()
}


# =========================================================
# Print results
# =========================================================

print("========== FULL V5 SPLIT ==========")

print("Train:", len(train_full))
print("Validation:", len(val_full))
print("Test:", len(test_full))

print(
    "Total:",
    len(train_full)
    + len(val_full)
    + len(test_full)
)


print("\n========== UNIQUE SENTENCES ==========")

print(
    "Train:",
    train_full["sentence"].nunique()
)

print(
    "Validation:",
    val_full["sentence"].nunique()
)

print(
    "Test:",
    test_full["sentence"].nunique()
)


print("\n========== OOV ==========")

print(
    "Validation OOV:",
    len(
        val_vocab_full
        -
        train_vocab_full
    )
)

print(
    "Test OOV:",
    len(
        test_vocab_full
        -
        train_vocab_full
    )
)


print("\n========== SENTENCE LEAKAGE ==========")

print(
    "Train-Val:",
    len(
        set(train_full["sentence"])
        &
        set(val_full["sentence"])
    )
)

print(
    "Train-Test:",
    len(
        set(train_full["sentence"])
        &
        set(test_full["sentence"])
    )
)

print(
    "Val-Test:",
    len(
        set(val_full["sentence"])
        &
        set(test_full["sentence"])
    )
)


# =========================================================
# Save split to Google Drive
# =========================================================

FULL_SPLIT_DIR = Path(
    "/content/drive/MyDrive/CSE499B/split_v5_full"
)

FULL_SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

train_full.to_csv(
    FULL_SPLIT_DIR / "train_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

val_full.to_csv(
    FULL_SPLIT_DIR / "val_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

test_full.to_csv(
    FULL_SPLIT_DIR / "test_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nSaved to:",
    FULL_SPLIT_DIR
)

========== FULL V5 SPLIT ==========
Train: 3506
Validation: 752
Test: 752
Total: 5010

========== UNIQUE SENTENCES ==========
Train: 2326
Validation: 533
Test: 519

========== OOV ==========
Validation OOV: 0
Test OOV: 0

========== SENTENCE LEAKAGE ==========
Train-Val: 0
Train-Test: 0
Val-Test: 0

Saved to: /content/drive/MyDrive/CSE499B/split_v5_full


#Prepare the Full V5 Training Pipeline

In [23]:
from collections import Counter
import numpy as np
import torch
from torch.utils.data import DataLoader

# =========================================================
# 1. FULL TRAIN VOCABULARY
# =========================================================

train_word_counter = Counter()

for sentence in train_full["sentence"]:
    train_word_counter.update(sentence.split())

train_words = sorted(
    train_word_counter.keys()
)

# CTC vocabulary
ctc_word_to_id = {
    "<blank>": 0
}

for word in train_words:
    ctc_word_to_id[word] = len(
        ctc_word_to_id
    )

ctc_id_to_word = {
    i: w
    for w, i in ctc_word_to_id.items()
}

CTC_BLANK_ID = 0


# Attention vocabulary
attn_word_to_id = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "<UNK>": 3
}

for word in train_words:
    attn_word_to_id[word] = len(
        attn_word_to_id
    )

attn_id_to_word = {
    i: w
    for w, i in attn_word_to_id.items()
}

PAD_ID = 0
SOS_ID = 1
EOS_ID = 2
UNK_ID = 3


# =========================================================
# 2. TRAIN-ONLY NORMALIZATION STATISTICS
# =========================================================

FEATURE_DIM = 258

feature_sum = np.zeros(
    FEATURE_DIM,
    dtype=np.float64
)

feature_sq_sum = np.zeros(
    FEATURE_DIM,
    dtype=np.float64
)

total_frames = 0


for idx in train_full["index"]:

    idx = clean_index(idx)

    x = np.load(
        keypoint_map[idx]
    ).astype(np.float64)

    feature_sum += x.sum(
        axis=0
    )

    feature_sq_sum += (
        x ** 2
    ).sum(
        axis=0
    )

    total_frames += x.shape[0]


TRAIN_MEAN = (
    feature_sum /
    total_frames
)

variance = (
    feature_sq_sum /
    total_frames
) - TRAIN_MEAN ** 2

variance = np.maximum(
    variance,
    0
)

TRAIN_STD = np.sqrt(
    variance
)

TRAIN_STD[
    TRAIN_STD < 1e-6
] = 1.0


TRAIN_MEAN_F32 = (
    TRAIN_MEAN.astype(
        np.float32
    )
)

TRAIN_STD_F32 = (
    TRAIN_STD.astype(
        np.float32
    )
)


# =========================================================
# 3. FULL DATASETS
# =========================================================

train_dataset = HybridSignDataset(
    train_full,
    training=True
)

val_dataset = HybridSignDataset(
    val_full,
    training=False
)

test_dataset = HybridSignDataset(
    test_full,
    training=False
)


# =========================================================
# 4. FULL DATALOADERS
# =========================================================

BATCH_SIZE = 32


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=hybrid_collate,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=hybrid_collate,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=hybrid_collate,
    num_workers=2,
    pin_memory=True
)


# =========================================================
# 5. FRESH FULL V5 MODEL
# =========================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = HybridCTCAttentionModel(
    ctc_vocab_size=len(
        ctc_word_to_id
    ),
    attn_vocab_size=len(
        attn_word_to_id
    )
).to(device)


# =========================================================
# SUMMARY
# =========================================================

print(
    "========== FULL V5 PIPELINE READY =========="
)

print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)

print(
    "Test samples:",
    len(test_dataset)
)

print(
    "Bangla training words:",
    len(train_words)
)

print(
    "CTC vocabulary:",
    len(ctc_word_to_id)
)

print(
    "Attention vocabulary:",
    len(attn_word_to_id)
)

print(
    "Training frames used for normalization:",
    total_frames
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Device:",
    device
)

print(
    "Model parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

========== FULL V5 PIPELINE READY ==========
Train samples: 3506
Validation samples: 752
Test samples: 752
Bangla training words: 1398
CTC vocabulary: 1399
Attention vocabulary: 1402
Training frames used for normalization: 112192
Batch size: 32
Device: cuda
Model parameters: 9763934


##Pretrain the Shared Encoder with CTC

Before joint CTC + attention training, the shared CNN-BiLSTM encoder is first trained using only the CTC objective.



In [24]:
import torch
import torch.nn as nn
import pandas as pd
from pathlib import Path

# =========================================================
# CONFIG
# =========================================================

CTC_PRETRAIN_EPOCHS = 15
CTC_LR = 3e-4
GRAD_CLIP = 1.0

CTC_PRETRAIN_DIR = Path(
    "/content/drive/MyDrive/CSE499B/v5_ctc_pretrain"
)

CTC_PRETRAIN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_CTC_PATH = (
    CTC_PRETRAIN_DIR /
    "best_ctc_pretrained.pt"
)


# =========================================================
# FREEZE ATTENTION DECODER
# =========================================================

for p in model.decoder.parameters():
    p.requires_grad = False

for p in model.encoder.parameters():
    p.requires_grad = True

for p in model.ctc_head.parameters():
    p.requires_grad = True


# =========================================================
# LOSS + OPTIMIZER
# =========================================================

ctc_criterion = nn.CTCLoss(
    blank=CTC_BLANK_ID,
    zero_infinity=True
)

ctc_optimizer = torch.optim.AdamW(
    list(model.encoder.parameters())
    +
    list(model.ctc_head.parameters()),
    lr=CTC_LR,
    weight_decay=1e-4
)

ctc_scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        ctc_optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )
)


# =========================================================
# EDIT DISTANCE
# =========================================================

def edit_distance(ref, hyp):

    n = len(ref)
    m = len(hyp)

    dp = [
        [0] * (m + 1)
        for _ in range(n + 1)
    ]

    for i in range(n + 1):
        dp[i][0] = i

    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):

        for j in range(1, m + 1):

            if ref[i - 1] == hyp[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]

            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],
                    dp[i][j - 1],
                    dp[i - 1][j - 1]
                )

    return dp[n][m]


# =========================================================
# CTC GREEDY DECODER
# =========================================================

@torch.no_grad()
def decode_ctc(
    ctc_log_probs,
    input_lengths
):

    # [T,B,C] -> [B,T]
    pred_ids = (
        ctc_log_probs
        .argmax(dim=-1)
        .transpose(0, 1)
    )

    decoded = []

    for i in range(
        pred_ids.size(0)
    ):

        seq = pred_ids[
            i,
            :input_lengths[i]
        ].tolist()

        collapsed = []

        prev = None

        for token in seq:

            if token != prev:
                collapsed.append(token)

            prev = token

        words = [
            ctc_id_to_word[token]
            for token in collapsed
            if token != CTC_BLANK_ID
        ]

        decoded.append(words)

    return decoded


# =========================================================
# CTC VALIDATION
# =========================================================

@torch.no_grad()
def evaluate_ctc(loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    total_edits = 0
    total_words = 0

    exact_correct = 0


    for batch in loader:

        features = batch[
            "features"
        ].to(device)

        input_lengths = batch[
            "input_lengths"
        ]

        targets = batch[
            "ctc_targets"
        ].to(device)

        target_lengths = batch[
            "ctc_target_lengths"
        ]


        encoded = model.encoder(
            features,
            input_lengths
        )

        logits = model.ctc_head(
            encoded
        )

        log_probs = torch.log_softmax(
            logits,
            dim=-1
        ).transpose(0, 1)


        loss = ctc_criterion(
            log_probs,
            targets,
            input_lengths,
            target_lengths
        )


        batch_size_now = (
            features.size(0)
        )

        total_loss += (
            loss.item()
            *
            batch_size_now
        )

        total_samples += (
            batch_size_now
        )


        predictions = decode_ctc(
            log_probs,
            input_lengths
        )


        for reference_sentence, prediction in zip(
            batch["sentences"],
            predictions
        ):

            reference = (
                reference_sentence.split()
            )

            total_edits += edit_distance(
                reference,
                prediction
            )

            total_words += len(
                reference
            )

            if reference == prediction:
                exact_correct += 1


    return {
        "loss":
            total_loss / total_samples,

        "wer":
            total_edits / total_words,

        "accuracy":
            exact_correct / total_samples
    }


# =========================================================
# PRETRAIN
# =========================================================

best_ctc_wer = float("inf")

ctc_history = []


print(
    "=========================================="
)

print(
    "FULL V5 CTC PRETRAINING"
)

print(
    "=========================================="
)

print(
    "Train:",
    len(train_dataset)
)

print(
    "Validation:",
    len(val_dataset)
)

print(
    "Epochs:",
    CTC_PRETRAIN_EPOCHS
)

print()


for epoch in range(
    1,
    CTC_PRETRAIN_EPOCHS + 1
):

    model.train()

    # Keep decoder frozen
    model.decoder.eval()

    running_loss = 0.0
    seen = 0


    for batch in train_loader:

        features = batch[
            "features"
        ].to(
            device,
            non_blocking=True
        )

        input_lengths = batch[
            "input_lengths"
        ]

        targets = batch[
            "ctc_targets"
        ].to(
            device,
            non_blocking=True
        )

        target_lengths = batch[
            "ctc_target_lengths"
        ]


        ctc_optimizer.zero_grad()


        encoded = model.encoder(
            features,
            input_lengths
        )

        logits = model.ctc_head(
            encoded
        )

        log_probs = torch.log_softmax(
            logits,
            dim=-1
        ).transpose(0, 1)


        loss = ctc_criterion(
            log_probs,
            targets,
            input_lengths,
            target_lengths
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            list(model.encoder.parameters())
            +
            list(model.ctc_head.parameters()),
            GRAD_CLIP
        )


        ctc_optimizer.step()


        batch_size_now = (
            features.size(0)
        )

        running_loss += (
            loss.item()
            *
            batch_size_now
        )

        seen += batch_size_now


    train_loss = (
        running_loss /
        seen
    )


    val_result = evaluate_ctc(
        val_loader
    )


    ctc_scheduler.step(
        val_result["wer"]
    )


    current_lr = (
        ctc_optimizer
        .param_groups[0]["lr"]
    )


    print(
        f"Epoch {epoch:02d}/{CTC_PRETRAIN_EPOCHS} | "
        f"Train Loss={train_loss:.4f} | "
        f"Val Loss={val_result['loss']:.4f} | "
        f"Val WER={val_result['wer']:.4f} | "
        f"Exact={val_result['accuracy']:.4f} | "
        f"LR={current_lr:.6f}"
    )


    ctc_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_result["loss"],
        "val_wer": val_result["wer"],
        "val_accuracy": val_result["accuracy"],
        "lr": current_lr
    })


    if val_result["wer"] < best_ctc_wer:

        best_ctc_wer = (
            val_result["wer"]
        )

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "best_val_wer":
                    best_ctc_wer,

                "ctc_word_to_id":
                    ctc_word_to_id,

                "attn_word_to_id":
                    attn_word_to_id,

                "train_mean":
                    TRAIN_MEAN_F32,

                "train_std":
                    TRAIN_STD_F32
            },

            BEST_CTC_PATH
        )

        print(
            f"   ✅ Best CTC model saved "
            f"(WER={best_ctc_wer:.4f})"
        )


# =========================================================
# SAVE HISTORY
# =========================================================

pd.DataFrame(
    ctc_history
).to_csv(
    CTC_PRETRAIN_DIR /
    "ctc_pretrain_history.csv",
    index=False
)


print(
    "\nBest validation CTC WER:",
    best_ctc_wer
)

print(
    "Saved at:",
    BEST_CTC_PATH
)

FULL V5 CTC PRETRAINING
Train: 3506
Validation: 752
Epochs: 15

Epoch 01/15 | Train Loss=9.3878 | Val Loss=5.6034 | Val WER=1.0000 | Exact=0.0000 | LR=0.000300
   ✅ Best CTC model saved (WER=1.0000)
Epoch 02/15 | Train Loss=5.7160 | Val Loss=5.0955 | Val WER=0.9534 | Exact=0.0000 | LR=0.000300
   ✅ Best CTC model saved (WER=0.9534)
Epoch 03/15 | Train Loss=5.3050 | Val Loss=4.7648 | Val WER=0.8450 | Exact=0.0000 | LR=0.000300
   ✅ Best CTC model saved (WER=0.8450)
Epoch 04/15 | Train Loss=4.9841 | Val Loss=4.5100 | Val WER=0.8371 | Exact=0.0000 | LR=0.000300
   ✅ Best CTC model saved (WER=0.8371)
Epoch 05/15 | Train Loss=4.7251 | Val Loss=4.3058 | Val WER=0.7870 | Exact=0.0000 | LR=0.000300
   ✅ Best CTC model saved (WER=0.7870)
Epoch 06/15 | Train Loss=4.5114 | Val Loss=4.1783 | Val WER=0.7949 | Exact=0.0000 | LR=0.000300
Epoch 07/15 | Train Loss=4.3478 | Val Loss=4.0622 | Val WER=0.7632 | Exact=0.0000 | LR=0.000300
   ✅ Best CTC model saved (WER=0.7632)
Epoch 08/15 | Train Loss=4.204

In [25]:
# =========================================================
# CONTINUE CTC PRETRAINING: EPOCH 16 -> 25
# =========================================================

START_EPOCH = 16
END_EPOCH = 25


for epoch in range(
    START_EPOCH,
    END_EPOCH + 1
):

    model.train()

    # Attention decoder remains frozen
    model.decoder.eval()

    running_loss = 0.0
    seen = 0


    for batch in train_loader:

        features = batch[
            "features"
        ].to(
            device,
            non_blocking=True
        )

        input_lengths = batch[
            "input_lengths"
        ]

        targets = batch[
            "ctc_targets"
        ].to(
            device,
            non_blocking=True
        )

        target_lengths = batch[
            "ctc_target_lengths"
        ]


        ctc_optimizer.zero_grad()


        # -----------------------------
        # Encoder
        # -----------------------------

        encoded = model.encoder(
            features,
            input_lengths
        )


        # -----------------------------
        # CTC Head
        # -----------------------------

        logits = model.ctc_head(
            encoded
        )

        log_probs = torch.log_softmax(
            logits,
            dim=-1
        ).transpose(
            0,
            1
        )


        # -----------------------------
        # CTC Loss
        # -----------------------------

        loss = ctc_criterion(
            log_probs,
            targets,
            input_lengths,
            target_lengths
        )


        # -----------------------------
        # Backpropagation
        # -----------------------------

        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            list(
                model.encoder.parameters()
            )
            +
            list(
                model.ctc_head.parameters()
            ),
            GRAD_CLIP
        )


        ctc_optimizer.step()


        batch_size_now = (
            features.size(0)
        )

        running_loss += (
            loss.item()
            *
            batch_size_now
        )

        seen += batch_size_now


    train_loss = (
        running_loss /
        seen
    )


    # -----------------------------
    # Validation
    # -----------------------------

    val_result = evaluate_ctc(
        val_loader
    )


    ctc_scheduler.step(
        val_result["wer"]
    )


    current_lr = (
        ctc_optimizer
        .param_groups[0]["lr"]
    )


    print(
        f"Epoch {epoch:02d}/{END_EPOCH} | "
        f"Train Loss={train_loss:.4f} | "
        f"Val Loss={val_result['loss']:.4f} | "
        f"Val WER={val_result['wer']:.4f} | "
        f"Exact={val_result['accuracy']:.4f} | "
        f"LR={current_lr:.6f}"
    )


    ctc_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_result["loss"],
        "val_wer": val_result["wer"],
        "val_accuracy": val_result["accuracy"],
        "lr": current_lr
    })


    # -----------------------------
    # Save best model
    # -----------------------------

    if val_result["wer"] < best_ctc_wer:

        best_ctc_wer = (
            val_result["wer"]
        )


        torch.save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "best_val_wer":
                    best_ctc_wer,

                "ctc_word_to_id":
                    ctc_word_to_id,

                "attn_word_to_id":
                    attn_word_to_id,

                "train_mean":
                    TRAIN_MEAN_F32,

                "train_std":
                    TRAIN_STD_F32
            },

            BEST_CTC_PATH
        )


        print(
            f"   ✅ Best CTC model saved "
            f"(WER={best_ctc_wer:.4f})"
        )


# =========================================================
# SAVE UPDATED HISTORY
# =========================================================

pd.DataFrame(
    ctc_history
).to_csv(
    CTC_PRETRAIN_DIR /
    "ctc_pretrain_history.csv",
    index=False
)


print(
    "\nBest validation CTC WER:",
    best_ctc_wer
)

print(
    "Best model:",
    BEST_CTC_PATH
)

Epoch 16/25 | Train Loss=3.3140 | Val Loss=3.4748 | Val WER=0.6732 | Exact=0.0027 | LR=0.000300
Epoch 17/25 | Train Loss=3.2143 | Val Loss=3.4059 | Val WER=0.6637 | Exact=0.0013 | LR=0.000300
Epoch 18/25 | Train Loss=3.1234 | Val Loss=3.4243 | Val WER=0.6599 | Exact=0.0053 | LR=0.000300
   ✅ Best CTC model saved (WER=0.6599)
Epoch 19/25 | Train Loss=3.0264 | Val Loss=3.3169 | Val WER=0.6456 | Exact=0.0093 | LR=0.000300
   ✅ Best CTC model saved (WER=0.6456)
Epoch 20/25 | Train Loss=2.9373 | Val Loss=3.3603 | Val WER=0.6441 | Exact=0.0066 | LR=0.000300
   ✅ Best CTC model saved (WER=0.6441)
Epoch 21/25 | Train Loss=2.8523 | Val Loss=3.3010 | Val WER=0.6314 | Exact=0.0093 | LR=0.000300
   ✅ Best CTC model saved (WER=0.6314)
Epoch 22/25 | Train Loss=2.7548 | Val Loss=3.2845 | Val WER=0.6358 | Exact=0.0093 | LR=0.000300
Epoch 23/25 | Train Loss=2.6811 | Val Loss=3.2633 | Val WER=0.6326 | Exact=0.0066 | LR=0.000300
Epoch 24/25 | Train Loss=2.5898 | Val Loss=3.2406 | Val WER=0.6269 | Exact=0

###Joint CTC + Temporal-Attention Training

The best CTC-pretrained encoder is now used to initialize the final hybrid model.

During this stage:
- CNN + BiLSTM encoder remains trainable.
- CTC head remains active to preserve temporal alignment.
- Temporal Attention + LSTM decoder is trained for Bangla sentence generation.
- CTC and Attention losses are optimized jointly.
- Both CTC WER and Attention WER are monitored separately.
- The final checkpoint is selected using validation Attention WER.

The test set remains untouched until model selection is complete.

In [26]:
# =========================================================
# FINAL JOINT HYBRID TRAINING
# =========================================================

import torch
import torch.nn as nn
import pandas as pd
from pathlib import Path


# =========================================================
# 1. LOAD BEST CTC-PRETRAINED MODEL
# =========================================================

checkpoint = torch.load(
    BEST_CTC_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print(
    "Loaded CTC pretrained epoch:",
    checkpoint["epoch"]
)

print(
    "Loaded CTC validation WER:",
    checkpoint["best_val_wer"]
)


# =========================================================
# 2. UNFREEZE EVERYTHING
# =========================================================

for p in model.parameters():
    p.requires_grad = True


# =========================================================
# 3. CONFIG
# =========================================================

HYBRID_EPOCHS = 20

CTC_WEIGHT = 0.50
ATTN_WEIGHT = 0.50

ENCODER_LR = 1e-4
DECODER_LR = 3e-4

GRAD_CLIP = 1.0


HYBRID_DIR = Path(
    "/content/drive/MyDrive/CSE499B/v5_hybrid_full"
)

HYBRID_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_HYBRID_PATH = (
    HYBRID_DIR /
    "best_v5_hybrid_full.pt"
)


# =========================================================
# 4. LOSSES
# =========================================================

ctc_criterion = nn.CTCLoss(
    blank=CTC_BLANK_ID,
    zero_infinity=True
)

attn_criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)


# =========================================================
# 5. DIFFERENT LEARNING RATES
# =========================================================

optimizer = torch.optim.AdamW(

    [
        {
            "params":
                model.encoder.parameters(),
            "lr":
                ENCODER_LR
        },

        {
            "params":
                model.ctc_head.parameters(),
            "lr":
                ENCODER_LR
        },

        {
            "params":
                model.decoder.parameters(),
            "lr":
                DECODER_LR
        }
    ],

    weight_decay=1e-4
)


scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )
)


# =========================================================
# 6. HYBRID VALIDATION
# =========================================================

@torch.no_grad()
def evaluate_joint(loader):

    model.eval()

    total_loss = 0
    total_ctc_loss = 0
    total_attn_loss = 0
    total_samples = 0

    ctc_edits = 0
    attn_edits = 0

    total_words = 0

    ctc_exact = 0
    attn_exact = 0


    for batch in loader:

        features = batch[
            "features"
        ].to(device)

        input_lengths = batch[
            "input_lengths"
        ]

        ctc_targets = batch[
            "ctc_targets"
        ].to(device)

        ctc_target_lengths = batch[
            "ctc_target_lengths"
        ]

        attn_targets = batch[
            "attn_targets"
        ].to(device)


        decoder_inputs = (
            attn_targets[:, :-1]
        )

        decoder_targets = (
            attn_targets[:, 1:]
        )


        # -----------------------------------------
        # Forward
        # -----------------------------------------

        ctc_log_probs, attn_logits = model(
            features,
            input_lengths,
            decoder_inputs
        )


        # -----------------------------------------
        # Loss
        # -----------------------------------------

        ctc_loss = ctc_criterion(
            ctc_log_probs,
            ctc_targets,
            input_lengths,
            ctc_target_lengths
        )


        attn_loss = attn_criterion(

            attn_logits.reshape(
                -1,
                attn_logits.size(-1)
            ),

            decoder_targets.reshape(-1)
        )


        loss = (
            CTC_WEIGHT * ctc_loss
            +
            ATTN_WEIGHT * attn_loss
        )


        B = features.size(0)

        total_loss += loss.item() * B
        total_ctc_loss += ctc_loss.item() * B
        total_attn_loss += attn_loss.item() * B

        total_samples += B


        # =========================================
        # CTC predictions
        # =========================================

        ctc_predictions = decode_ctc(
            ctc_log_probs,
            input_lengths
        )


        # =========================================
        # Attention autoregressive predictions
        # =========================================

        attn_predictions = attention_greedy_decode(
            model,
            features,
            input_lengths,
            max_steps=ATTENTION_MAX_LENGTH - 1
        )


        # =========================================
        # Metrics
        # =========================================

        for ref_sentence, ctc_pred, attn_pred in zip(

            batch["sentences"],
            ctc_predictions,
            attn_predictions
        ):

            ref = ref_sentence.split()

            ctc_edits += edit_distance(
                ref,
                ctc_pred
            )

            attn_edits += edit_distance(
                ref,
                attn_pred
            )

            total_words += len(ref)


            if ref == ctc_pred:
                ctc_exact += 1

            if ref == attn_pred:
                attn_exact += 1


    return {

        "loss":
            total_loss / total_samples,

        "ctc_loss":
            total_ctc_loss / total_samples,

        "attn_loss":
            total_attn_loss / total_samples,

        "ctc_wer":
            ctc_edits / total_words,

        "attn_wer":
            attn_edits / total_words,

        "ctc_exact":
            ctc_exact / total_samples,

        "attn_exact":
            attn_exact / total_samples
    }


# =========================================================
# 7. TRAIN
# =========================================================

best_attn_wer = float("inf")

hybrid_history = []


print(
    "\n=============================================="
)

print(
    "FULL V5 JOINT CTC + ATTENTION TRAINING"
)

print(
    "=============================================="
)

print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)

print(
    "Epochs:",
    HYBRID_EPOCHS
)

print(
    "CTC weight:",
    CTC_WEIGHT
)

print(
    "Attention weight:",
    ATTN_WEIGHT
)

print()


for epoch in range(
    1,
    HYBRID_EPOCHS + 1
):

    model.train()

    running_total = 0
    running_ctc = 0
    running_attn = 0

    seen = 0


    for batch in train_loader:

        features = batch[
            "features"
        ].to(
            device,
            non_blocking=True
        )

        input_lengths = batch[
            "input_lengths"
        ]

        ctc_targets = batch[
            "ctc_targets"
        ].to(
            device,
            non_blocking=True
        )

        ctc_target_lengths = batch[
            "ctc_target_lengths"
        ]

        attn_targets = batch[
            "attn_targets"
        ].to(
            device,
            non_blocking=True
        )


        decoder_inputs = (
            attn_targets[:, :-1]
        )

        decoder_targets = (
            attn_targets[:, 1:]
        )


        optimizer.zero_grad()


        # -----------------------------------------
        # Forward
        # -----------------------------------------

        ctc_log_probs, attn_logits = model(
            features,
            input_lengths,
            decoder_inputs
        )


        # -----------------------------------------
        # CTC loss
        # -----------------------------------------

        ctc_loss = ctc_criterion(
            ctc_log_probs,
            ctc_targets,
            input_lengths,
            ctc_target_lengths
        )


        # -----------------------------------------
        # Attention loss
        # -----------------------------------------

        attn_loss = attn_criterion(

            attn_logits.reshape(
                -1,
                attn_logits.size(-1)
            ),

            decoder_targets.reshape(-1)
        )


        # -----------------------------------------
        # Joint loss
        # -----------------------------------------

        loss = (
            CTC_WEIGHT * ctc_loss
            +
            ATTN_WEIGHT * attn_loss
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )


        optimizer.step()


        B = features.size(0)

        running_total += (
            loss.item() * B
        )

        running_ctc += (
            ctc_loss.item() * B
        )

        running_attn += (
            attn_loss.item() * B
        )

        seen += B


    train_loss = (
        running_total / seen
    )

    train_ctc = (
        running_ctc / seen
    )

    train_attn = (
        running_attn / seen
    )


    # =====================================================
    # Validation
    # =====================================================

    val = evaluate_joint(
        val_loader
    )


    scheduler.step(
        val["attn_wer"]
    )


    encoder_lr = optimizer.param_groups[
        0
    ]["lr"]

    decoder_lr = optimizer.param_groups[
        2
    ]["lr"]


    print(

        f"Epoch {epoch:02d}/{HYBRID_EPOCHS} | "

        f"Train={train_loss:.4f} | "

        f"CTC={train_ctc:.4f} | "

        f"Attn={train_attn:.4f} | "

        f"Val={val['loss']:.4f} | "

        f"CTC-WER={val['ctc_wer']:.4f} | "

        f"ATTN-WER={val['attn_wer']:.4f} | "

        f"ATTN-Exact={val['attn_exact']:.4f} | "

        f"LR-E={encoder_lr:.6f} | "

        f"LR-D={decoder_lr:.6f}"
    )


    hybrid_history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "train_ctc":
            train_ctc,

        "train_attn":
            train_attn,

        "val_loss":
            val["loss"],

        "val_ctc_wer":
            val["ctc_wer"],

        "val_attn_wer":
            val["attn_wer"],

        "val_ctc_exact":
            val["ctc_exact"],

        "val_attn_exact":
            val["attn_exact"]
    })


    # =====================================================
    # SAVE BEST ATTENTION MODEL
    # =====================================================

    if (
        val["attn_wer"]
        <
        best_attn_wer
    ):

        best_attn_wer = (
            val["attn_wer"]
        )


        torch.save(

            {
                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "best_attn_wer":
                    best_attn_wer,

                "val_ctc_wer":
                    val["ctc_wer"],

                "ctc_word_to_id":
                    ctc_word_to_id,

                "attn_word_to_id":
                    attn_word_to_id,

                "train_mean":
                    TRAIN_MEAN_F32,

                "train_std":
                    TRAIN_STD_F32,

                "ctc_weight":
                    CTC_WEIGHT,

                "attn_weight":
                    ATTN_WEIGHT
            },

            BEST_HYBRID_PATH
        )


        print(
            f"   ✅ Best hybrid model saved "
            f"(Attention WER={best_attn_wer:.4f})"
        )


# =========================================================
# SAVE HISTORY
# =========================================================

pd.DataFrame(
    hybrid_history
).to_csv(

    HYBRID_DIR /
    "hybrid_training_history.csv",

    index=False
)


print(
    "\nBest Validation Attention WER:",
    best_attn_wer
)

print(
    "Best hybrid model:",
    BEST_HYBRID_PATH
)

Loaded CTC pretrained epoch: 25
Loaded CTC validation WER: 0.6225039619651347

FULL V5 JOINT CTC + ATTENTION TRAINING
Train samples: 3506
Validation samples: 752
Epochs: 20
CTC weight: 0.5
Attention weight: 0.5

Epoch 01/20 | Train=3.3630 | CTC=2.3765 | Attn=4.3494 | Val=3.1772 | CTC-WER=0.6165 | ATTN-WER=0.7055 | ATTN-Exact=0.0000 | LR-E=0.000100 | LR-D=0.000300
   ✅ Best hybrid model saved (Attention WER=0.7055)
Epoch 02/20 | Train=2.7248 | CTC=2.2972 | Attn=3.1523 | Val=2.9358 | CTC-WER=0.6158 | ATTN-WER=0.6995 | ATTN-Exact=0.0027 | LR-E=0.000100 | LR-D=0.000300
   ✅ Best hybrid model saved (Attention WER=0.6995)
Epoch 03/20 | Train=2.4313 | CTC=2.2477 | Attn=2.6149 | Val=2.7948 | CTC-WER=0.6063 | ATTN-WER=0.6732 | ATTN-Exact=0.0053 | LR-E=0.000100 | LR-D=0.000300
   ✅ Best hybrid model saved (Attention WER=0.6732)
Epoch 04/20 | Train=2.2133 | CTC=2.2099 | Attn=2.2166 | Val=2.7192 | CTC-WER=0.6048 | ATTN-WER=0.6748 | ATTN-Exact=0.0199 | LR-E=0.000100 | LR-D=0.000300
Epoch 05/20 | Tr

#Final Evaluation of the Best V5 Hybrid Model


In [27]:
import json
import pandas as pd
import torch
from pathlib import Path


# =========================================================
# 1. RECALCULATE FULL-DATA MAXIMUM TARGET LENGTH
# =========================================================

FULL_MAX_WORDS = max(
    train_full["sentence"].apply(
        lambda x: len(x.split())
    ).max(),

    val_full["sentence"].apply(
        lambda x: len(x.split())
    ).max(),

    test_full["sentence"].apply(
        lambda x: len(x.split())
    ).max()
)

# Maximum generated words + EOS
FULL_MAX_DECODE_STEPS = (
    FULL_MAX_WORDS + 1
)

print(
    "Full dataset maximum sentence words:",
    FULL_MAX_WORDS
)

print(
    "Attention maximum decoding steps:",
    FULL_MAX_DECODE_STEPS
)


# =========================================================
# 2. LOAD BEST HYBRID CHECKPOINT
# =========================================================

best_checkpoint = torch.load(
    BEST_HYBRID_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)

model.eval()


print(
    "\nLoaded best hybrid epoch:",
    best_checkpoint["epoch"]
)

print(
    "Best Validation Attention WER:",
    best_checkpoint[
        "best_attn_wer"
    ]
)

print(
    "Validation CTC WER at that epoch:",
    best_checkpoint[
        "val_ctc_wer"
    ]
)


# =========================================================
# 3. FINAL TEST EVALUATION
# =========================================================

@torch.no_grad()
def final_test_evaluation(loader):

    model.eval()

    total_ctc_edits = 0
    total_attn_edits = 0
    total_reference_words = 0

    ctc_exact = 0
    attn_exact = 0

    total_samples = 0

    prediction_rows = []


    for batch in loader:

        features = batch[
            "features"
        ].to(device)

        input_lengths = batch[
            "input_lengths"
        ]


        # -----------------------------------------
        # Shared encoder
        # -----------------------------------------

        encoded = model.encoder(
            features,
            input_lengths
        )


        # =========================================
        # CTC prediction
        # =========================================

        ctc_logits = model.ctc_head(
            encoded
        )

        ctc_log_probs = torch.log_softmax(
            ctc_logits,
            dim=-1
        ).transpose(0, 1)


        ctc_predictions = decode_ctc(
            ctc_log_probs,
            input_lengths
        )


        # =========================================
        # Attention autoregressive prediction
        # =========================================

        attn_predictions = (
            attention_greedy_decode(
                model,
                features,
                input_lengths,
                max_steps=
                FULL_MAX_DECODE_STEPS
            )
        )


        # =========================================
        # Metrics
        # =========================================

        for (
            sample_id,
            reference_sentence,
            ctc_pred,
            attn_pred
        ) in zip(

            batch["sample_ids"],
            batch["sentences"],
            ctc_predictions,
            attn_predictions
        ):

            reference_words = (
                reference_sentence.split()
            )


            ctc_distance = edit_distance(
                reference_words,
                ctc_pred
            )

            attn_distance = edit_distance(
                reference_words,
                attn_pred
            )


            total_ctc_edits += (
                ctc_distance
            )

            total_attn_edits += (
                attn_distance
            )

            total_reference_words += len(
                reference_words
            )

            total_samples += 1


            if (
                reference_words
                ==
                ctc_pred
            ):
                ctc_exact += 1


            if (
                reference_words
                ==
                attn_pred
            ):
                attn_exact += 1


            prediction_rows.append({

                "sample_id":
                    sample_id,

                "reference":
                    reference_sentence,

                "ctc_prediction":
                    " ".join(
                        ctc_pred
                    ),

                "attention_prediction":
                    " ".join(
                        attn_pred
                    ),

                "ctc_edit_distance":
                    ctc_distance,

                "attention_edit_distance":
                    attn_distance
            })


    metrics = {

        "test_samples":
            total_samples,

        "ctc_wer":
            total_ctc_edits
            /
            total_reference_words,

        "attention_wer":
            total_attn_edits
            /
            total_reference_words,

        "ctc_exact_sentence_accuracy":
            ctc_exact
            /
            total_samples,

        "attention_exact_sentence_accuracy":
            attn_exact
            /
            total_samples,

        "best_hybrid_epoch":
            best_checkpoint[
                "epoch"
            ],

        "validation_attention_wer":
            best_checkpoint[
                "best_attn_wer"
            ],

        "validation_ctc_wer":
            best_checkpoint[
                "val_ctc_wer"
            ],

        "max_sentence_words":
            int(
                FULL_MAX_WORDS
            )
    }


    return (
        metrics,
        pd.DataFrame(
            prediction_rows
        )
    )


# =========================================================
# 4. RUN TEST ONLY ONCE
# =========================================================

final_metrics, prediction_df = (
    final_test_evaluation(
        test_loader
    )
)


# =========================================================
# 5. SAVE FINAL RESULTS
# =========================================================

FINAL_RESULT_DIR = Path(
    "/content/drive/MyDrive/CSE499B/v5_final_results"
)

FINAL_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


prediction_df.to_csv(
    FINAL_RESULT_DIR /
    "test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)


with open(
    FINAL_RESULT_DIR /
    "final_metrics.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_metrics,
        f,
        ensure_ascii=False,
        indent=4
    )


# =========================================================
# 6. PRINT FINAL METRICS
# =========================================================

print(
    "\n=============================================="
)

print(
    "FINAL V5 TEST RESULT"
)

print(
    "=============================================="
)

print(
    "Test samples:",
    final_metrics[
        "test_samples"
    ]
)

print(
    "CTC WER:",
    round(
        final_metrics[
            "ctc_wer"
        ],
        4
    )
)

print(
    "CTC Exact Sentence Accuracy:",
    round(
        final_metrics[
            "ctc_exact_sentence_accuracy"
        ],
        4
    )
)

print(
    "Attention WER:",
    round(
        final_metrics[
            "attention_wer"
        ],
        4
    )
)

print(
    "Attention Exact Sentence Accuracy:",
    round(
        final_metrics[
            "attention_exact_sentence_accuracy"
        ],
        4
    )
)


# =========================================================
# 7. SHOW 15 REAL BANGLA PREDICTIONS
# =========================================================

print(
    "\n=============================================="
)

print(
    "SAMPLE TEST PREDICTIONS"
)

print(
    "=============================================="
)


for i, row in (
    prediction_df
    .head(15)
    .iterrows()
):

    print(
        f"\nExample {i + 1}"
    )

    print(
        "Reference :",
        row["reference"]
    )

    print(
        "CTC       :",
        row["ctc_prediction"]
    )

    print(
        "Attention :",
        row[
            "attention_prediction"
        ]
    )


print(
    "\nSaved predictions:"
)

print(
    FINAL_RESULT_DIR /
    "test_predictions.csv"
)

print(
    "\nSaved metrics:"
)

print(
    FINAL_RESULT_DIR /
    "final_metrics.json"
)

Full dataset maximum sentence words: 13
Attention maximum decoding steps: 14

Loaded best hybrid epoch: 15
Best Validation Attention WER: 0.6076069730586371
Validation CTC WER at that epoch: 0.5816164817749604

FINAL V5 TEST RESULT
Test samples: 752
CTC WER: 0.5711
CTC Exact Sentence Accuracy: 0.0093
Attention WER: 0.5817
Attention Exact Sentence Accuracy: 0.0386

SAMPLE TEST PREDICTIONS

Example 1
Reference : আমার মা জমিতে যাবে
CTC       : আমার ভাইয়ের হাস ভালো লাগে
Attention : আমার ভাইয়ের মানচিত্র দেখতে ভালো লাগে

Example 2
Reference : আমার মা জমিতে যাবে
CTC       : আমার বাবা জমিতে যাবে
Attention : আমার বাবা বনে যাবে

Example 3
Reference : আমার মেয়ে বুধবার মাছ খায়
CTC       : আমার বৃহস্পতিবার মাছ খায়
Attention : আমার মেয়ে বৃহস্পতিবার মাছ খায়

Example 4
Reference : তুমি মংগলবার দুধ খাবে
CTC       : তুমি বুধবার মাংস খায়
Attention : তুমি বুধবার মাংস খাবে

Example 5
Reference : আমি ফেব্রুয়ারিতে ভারত যাব
CTC       : আমি মাসে বিদেশ যাবো
Attention : আমি ডিসেম্বরে জাপান যাবো

Example

##Joint CTC + Attention Beam Search on Validation Set


Instead of selecting only the single highest-probability word at every decoding step:

1. The attention decoder keeps multiple candidate sentences using beam search.
2. The CTC branch scores each candidate according to temporal sign alignment.
3. Attention and CTC scores are combined.
4. The best combination weight is selected using only the validation set.

This can improve decoding without additional model training and therefore does not increase model overfitting.

The test set is not used in this step.

In [28]:
import math
import numpy as np
import torch
import torch.nn.functional as F


# =========================================================
# CONFIG
# =========================================================

BEAM_SIZE = 5

MAX_DECODE_STEPS = FULL_MAX_DECODE_STEPS

# How much CTC contributes during reranking
CTC_WEIGHTS = [
    0.0,
    0.25,
    0.50,
    0.75,
    1.00
]


# =========================================================
# Convert attention token IDs -> CTC token IDs
# =========================================================

def attn_ids_to_ctc_ids(attn_ids):

    ctc_ids = []

    for token_id in attn_ids:

        word = attn_id_to_word[token_id]

        if word in {
            "<PAD>",
            "<SOS>",
            "<EOS>",
            "<UNK>"
        }:
            continue

        if word not in ctc_word_to_id:
            return None

        ctc_ids.append(
            ctc_word_to_id[word]
        )

    return ctc_ids


# =========================================================
# ATTENTION BEAM SEARCH — ONE SAMPLE
# =========================================================

@torch.no_grad()
def attention_beam_search_single(
    model,
    encoder_outputs,
    input_length,
    beam_size=5,
    max_steps=14
):

    decoder = model.decoder

    device_now = encoder_outputs.device

    T = encoder_outputs.size(1)


    # -----------------------------------------------------
    # Encoder mask
    # -----------------------------------------------------

    mask = (
        torch.arange(
            T,
            device=device_now
        ).unsqueeze(0)
        <
        torch.tensor(
            [[input_length]],
            device=device_now
        )
    )


    mask_float = (
        mask.unsqueeze(-1).float()
    )


    # -----------------------------------------------------
    # Initial decoder state
    # -----------------------------------------------------

    mean_encoded = (
        encoder_outputs *
        mask_float
    ).sum(dim=1)

    mean_encoded = (
        mean_encoded /
        float(input_length)
    )


    h0 = torch.tanh(
        decoder.init_h(
            mean_encoded
        )
    )

    c0 = torch.tanh(
        decoder.init_c(
            mean_encoded
        )
    )


    # beam:
    # tokens, score, h, c, finished

    beams = [
        (
            [SOS_ID],
            0.0,
            h0,
            c0,
            False
        )
    ]


    for _ in range(max_steps):

        new_beams = []


        for (
            tokens,
            score,
            h,
            c,
            finished
        ) in beams:

            if finished:

                new_beams.append(
                    (
                        tokens,
                        score,
                        h,
                        c,
                        True
                    )
                )

                continue


            current_token = torch.tensor(
                [tokens[-1]],
                dtype=torch.long,
                device=device_now
            )


            embedding = decoder.embedding(
                current_token
            )


            context, _ = decoder.attention(
                encoder_outputs,
                h,
                mask
            )


            decoder_input = torch.cat(
                [
                    embedding,
                    context
                ],
                dim=1
            )


            new_h, new_c = decoder.lstm(
                decoder_input,
                (h, c)
            )


            logits = decoder.output(
                torch.cat(
                    [
                        new_h,
                        context
                    ],
                    dim=1
                )
            )


            log_probs = F.log_softmax(
                logits,
                dim=-1
            )


            top_log_probs, top_ids = (
                log_probs.topk(
                    beam_size,
                    dim=-1
                )
            )


            for k in range(
                beam_size
            ):

                token_id = int(
                    top_ids[0, k].item()
                )

                token_score = float(
                    top_log_probs[
                        0,
                        k
                    ].item()
                )


                # Do not generate PAD/SOS
                if token_id in {
                    PAD_ID,
                    SOS_ID
                }:
                    continue


                new_tokens = (
                    tokens +
                    [token_id]
                )


                new_score = (
                    score +
                    token_score
                )


                is_finished = (
                    token_id
                    ==
                    EOS_ID
                )


                new_beams.append(
                    (
                        new_tokens,
                        new_score,
                        new_h,
                        new_c,
                        is_finished
                    )
                )


        # ---------------------------------------------
        # Length-normalized ranking
        # ---------------------------------------------

        def beam_rank(x):

            generated_len = max(
                len(x[0]) - 1,
                1
            )

            return (
                x[1]
                /
                (
                    generated_len
                    ** 0.7
                )
            )


        new_beams.sort(
            key=beam_rank,
            reverse=True
        )


        beams = new_beams[
            :beam_size
        ]


        if all(
            x[4]
            for x in beams
        ):
            break


    results = []


    for (
        tokens,
        score,
        h,
        c,
        finished
    ) in beams:

        # Remove SOS
        tokens = tokens[1:]


        # Stop at EOS
        cleaned = []

        for token in tokens:

            if token == EOS_ID:
                break

            if token not in {
                PAD_ID,
                SOS_ID
            }:
                cleaned.append(
                    token
                )


        length = max(
            len(cleaned),
            1
        )


        attention_score = (
            score /
            (
                length ** 0.7
            )
        )


        results.append(
            {
                "tokens":
                    cleaned,

                "attention_score":
                    attention_score
            }
        )


    return results


# =========================================================
# CTC SCORE FOR ONE CANDIDATE
# =========================================================

@torch.no_grad()
def get_ctc_candidate_score(
    log_probs_single,
    candidate_attn_ids,
    input_length
):

    ctc_ids = attn_ids_to_ctc_ids(
        candidate_attn_ids
    )


    if (
        ctc_ids is None
        or len(ctc_ids) == 0
    ):
        return -1e9


    target = torch.tensor(
        ctc_ids,
        dtype=torch.long,
        device=log_probs_single.device
    )


    input_len_tensor = torch.tensor(
        [input_length],
        dtype=torch.long
    )


    target_len_tensor = torch.tensor(
        [len(ctc_ids)],
        dtype=torch.long
    )


    # [T,C] -> [T,1,C]
    log_probs_for_loss = (
        log_probs_single[
            :input_length
        ]
        .unsqueeze(1)
    )


    loss = F.ctc_loss(
        log_probs_for_loss,
        target,
        input_len_tensor,
        target_len_tensor,
        blank=CTC_BLANK_ID,
        reduction="sum",
        zero_infinity=True
    )


    # Higher score = better
    return (
        -float(
            loss.item()
        )
        /
        max(
            len(ctc_ids),
            1
        )
    )


# =========================================================
# COLLECT VALIDATION N-BEST CANDIDATES
# =========================================================

@torch.no_grad()
def collect_validation_candidates():

    model.eval()

    all_items = []


    for batch in val_loader:

        features = batch[
            "features"
        ].to(device)

        lengths = batch[
            "input_lengths"
        ]


        encoded = model.encoder(
            features,
            lengths
        )


        ctc_logits = model.ctc_head(
            encoded
        )


        ctc_log_probs = F.log_softmax(
            ctc_logits,
            dim=-1
        )


        for i in range(
            features.size(0)
        ):

            input_len = int(
                lengths[i].item()
            )


            encoder_single = (
                encoded[
                    i:i+1
                ]
            )


            candidates = (
                attention_beam_search_single(
                    model,
                    encoder_single,
                    input_len,
                    beam_size=BEAM_SIZE,
                    max_steps=
                    MAX_DECODE_STEPS
                )
            )


            # -----------------------------------------
            # Add CTC score
            # -----------------------------------------

            for candidate in candidates:

                candidate[
                    "ctc_score"
                ] = (
                    get_ctc_candidate_score(
                        ctc_log_probs[i],
                        candidate[
                            "tokens"
                        ],
                        input_len
                    )
                )


            all_items.append(
                {
                    "reference":
                        batch[
                            "sentences"
                        ][i],

                    "candidates":
                        candidates
                }
            )


    return all_items


print(
    "Collecting validation beam candidates..."
)

validation_candidates = (
    collect_validation_candidates()
)

print(
    "Validation samples processed:",
    len(
        validation_candidates
    )
)


# =========================================================
# SCORE NORMALIZATION WITHIN EACH BEAM
# =========================================================

def normalize_scores(values):

    values = np.array(
        values,
        dtype=np.float64
    )

    std = values.std()

    if std < 1e-8:
        return np.zeros_like(
            values
        )

    return (
        values -
        values.mean()
    ) / std


# =========================================================
# TRY DIFFERENT CTC WEIGHTS ON VALIDATION ONLY
# =========================================================

results = []


for ctc_weight in CTC_WEIGHTS:

    total_edits = 0
    total_words = 0
    exact = 0


    for item in validation_candidates:

        reference = (
            item[
                "reference"
            ].split()
        )

        candidates = item[
            "candidates"
        ]


        attn_scores = [
            x["attention_score"]
            for x in candidates
        ]

        ctc_scores = [
            x["ctc_score"]
            for x in candidates
        ]


        attn_norm = normalize_scores(
            attn_scores
        )

        ctc_norm = normalize_scores(
            ctc_scores
        )


        combined_scores = (

            (1.0 - ctc_weight)
            *
            attn_norm

            +

            ctc_weight
            *
            ctc_norm
        )


        best_index = int(
            np.argmax(
                combined_scores
            )
        )


        best_tokens = candidates[
            best_index
        ]["tokens"]


        prediction = [
            attn_id_to_word[token]
            for token in best_tokens
        ]


        total_edits += edit_distance(
            reference,
            prediction
        )

        total_words += len(
            reference
        )


        if reference == prediction:
            exact += 1


    wer = (
        total_edits /
        total_words
    )

    accuracy = (
        exact /
        len(
            validation_candidates
        )
    )


    results.append(
        {
            "ctc_weight":
                ctc_weight,

            "validation_wer":
                wer,

            "exact_accuracy":
                accuracy
        }
    )


# =========================================================
# RESULTS
# =========================================================

result_df = pd.DataFrame(
    results
)


print(
    "\n=============================================="
)

print(
    "VALIDATION JOINT BEAM SEARCH"
)

print(
    "=============================================="
)

print(
    result_df.to_string(
        index=False
    )
)


best_row = result_df.loc[
    result_df[
        "validation_wer"
    ].idxmin()
]


BEST_BEAM_CTC_WEIGHT = float(
    best_row[
        "ctc_weight"
    ]
)


print(
    "\nBest CTC reranking weight:",
    BEST_BEAM_CTC_WEIGHT
)

print(
    "Best Validation WER:",
    round(
        float(
            best_row[
                "validation_wer"
            ]
        ),
        4
    )
)

print(
    "Exact Sentence Accuracy:",
    round(
        float(
            best_row[
                "exact_accuracy"
            ]
        ),
        4
    )
)

Validation samples processed: 752

VALIDATION JOINT BEAM SEARCH
 ctc_weight  validation_wer  exact_accuracy
       0.00        0.621553        0.018617
       0.25        0.624089        0.018617
       0.50        0.610143        0.021277
       0.75        0.593027        0.033245
       1.00        0.593027        0.034574

Best CTC reranking weight: 0.75
Best Validation WER: 0.593
Exact Sentence Accuracy: 0.0332


In [29]:
import zipfile
import shutil
import numpy as np
import torch
from pathlib import Path
from IPython.display import Video, display


# =========================================================
# 1. SELECT ONE TEST SAMPLE
# =========================================================

DEMO_INDEX = 2       # change: 0, 1, 2, 3, ...

row = test_full.iloc[DEMO_INDEX]

sample_id = clean_index(
    row["index"]
)

ground_truth = row["sentence"]


print("Sample ID:", sample_id)


# =========================================================
# 2. FIND RAW DATASET ZIP
# =========================================================

possible_zip_paths = [

    Path(
        "/content/drive/MyDrive/CSE499B/Sign_Language.zip"
    ),

    Path(
        "/content/drive/MyDrive/Sign_Language.zip"
    )
]


RAW_ZIP = None

for p in possible_zip_paths:

    if p.exists():
        RAW_ZIP = p
        break


if RAW_ZIP is None:

    raise FileNotFoundError(
        "Sign_Language.zip not found. "
        "Set RAW_ZIP to your actual Drive path."
    )


print(
    "Dataset ZIP:",
    RAW_ZIP
)


# =========================================================
# 3. EXTRACT ONLY THIS ONE VIDEO
#    (not the whole 65 GB dataset)
# =========================================================

DEMO_DIR = Path(
    "/content/demo_video"
)

DEMO_DIR.mkdir(
    exist_ok=True
)


with zipfile.ZipFile(
    RAW_ZIP,
    "r"
) as zf:

    members = zf.namelist()

    video_member = None

    for name in members:

        if name.endswith(
            f"/{sample_id}.mp4"
        ):

            video_member = name
            break


    if video_member is None:

        raise FileNotFoundError(
            f"Video {sample_id}.mp4 "
            f"not found inside ZIP."
        )


    extracted_path = Path(
        zf.extract(
            video_member,
            DEMO_DIR
        )
    )


demo_video_path = (
    DEMO_DIR /
    f"{sample_id}.mp4"
)


shutil.copy2(
    extracted_path,
    demo_video_path
)


print(
    "\nVideo extracted:",
    demo_video_path
)


# =========================================================
# 4. DISPLAY ORIGINAL TEST VIDEO
# =========================================================

display(
    Video(
        str(demo_video_path),
        embed=True,
        width=500
    )
)


# =========================================================
# 5. LOAD CORRESPONDING KEYPOINTS
# =========================================================

x = np.load(
    keypoint_map[sample_id]
).astype(
    np.float32
)


print(
    "\nRaw keypoint shape:",
    x.shape
)


# Same training normalization
x = (
    x - TRAIN_MEAN_F32
) / TRAIN_STD_F32


features = torch.tensor(
    x,
    dtype=torch.float32
).unsqueeze(0).to(device)


input_lengths = torch.tensor(
    [x.shape[0]],
    dtype=torch.long
)


# =========================================================
# 6. LOAD BEST HYBRID MODEL
# =========================================================

checkpoint = torch.load(
    BEST_HYBRID_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()


# =========================================================
# 7. CTC PREDICTION
# =========================================================

with torch.no_grad():

    encoded = model.encoder(
        features,
        input_lengths
    )

    ctc_logits = model.ctc_head(
        encoded
    )

    ctc_log_probs = torch.log_softmax(
        ctc_logits,
        dim=-1
    ).transpose(
        0,
        1
    )


ctc_prediction = decode_ctc(
    ctc_log_probs,
    input_lengths
)[0]


# =========================================================
# 8. ATTENTION GREEDY PREDICTION
# =========================================================

attention_prediction = (
    attention_greedy_decode(
        model,
        features,
        input_lengths,
        max_steps=FULL_MAX_DECODE_STEPS
    )[0]
)


# =========================================================
# 9. ATTENTION BEAM CANDIDATES
# =========================================================

with torch.no_grad():

    encoded = model.encoder(
        features,
        input_lengths
    )

    ctc_logits_bt = model.ctc_head(
        encoded
    )

    ctc_log_probs_bt = (
        torch.log_softmax(
            ctc_logits_bt,
            dim=-1
        )
    )


beam_candidates = (
    attention_beam_search_single(
        model,
        encoded,
        int(input_lengths[0]),
        beam_size=BEAM_SIZE,
        max_steps=FULL_MAX_DECODE_STEPS
    )
)


# =========================================================
# 10. CTC SCORE EACH ATTENTION CANDIDATE
# =========================================================

for candidate in beam_candidates:

    candidate["ctc_score"] = (
        get_ctc_candidate_score(
            ctc_log_probs_bt[0],
            candidate["tokens"],
            int(input_lengths[0])
        )
    )


attn_scores = [
    c["attention_score"]
    for c in beam_candidates
]

ctc_scores = [
    c["ctc_score"]
    for c in beam_candidates
]


attn_norm = normalize_scores(
    attn_scores
)

ctc_norm = normalize_scores(
    ctc_scores
)


# Validation-selected weight
beam_ctc_weight = (
    BEST_BEAM_CTC_WEIGHT
)


combined_scores = (

    (1.0 - beam_ctc_weight)
    * attn_norm

    +

    beam_ctc_weight
    * ctc_norm
)


best_candidate_index = int(
    np.argmax(
        combined_scores
    )
)


best_tokens = beam_candidates[
    best_candidate_index
]["tokens"]


joint_prediction = [

    attn_id_to_word[token]

    for token in best_tokens
]


# =========================================================
# 11. FINAL DEMO OUTPUT
# =========================================================

print(
    "\n=============================================="
)

print(
    "BANGLA SIGN LANGUAGE DEMO"
)

print(
    "=============================================="
)


print(
    "\nCTC Prediction:"
)

print(
    " ".join(
        ctc_prediction
    )
)


print(
    "\nAttention Prediction:"
)

print(
    " ".join(
        attention_prediction
    )
)


print(
    "\nFINAL JOINT-BEAM PREDICTION:"
)

print(
    " ".join(
        joint_prediction
    )
)


print(
    "\nGround Truth:"
)

print(
    ground_truth
)